# Two-layer cabinet clustering

Groups cabinets into **clusters**. A cluster is one day of field work for one team and becomes one card on the
manager's calendar. This notebook decides how many people each cluster needs; `WorkerLayer.ipynb` then picks the
workers.

**Rules every cluster keeps** (the numbers come from `parameters.csv`, see *Model rules*):

1. **Demand.** A team of *n* people takes between *n* × `MIN_DEMAND_PER_PERSON` and *n* × `DEMAND_PER_PERSON`
   tickets/day. With the usual 4 to 6 tickets per person and teams of up to 6, a full team takes 24 to 36 (`CAP`).
   A quiet cluster gets a smaller team, down to `MIN_PEOPLE`; one person may take up to `SOLO_MAX_DEMAND` as a
   special case.
2. **Distance.** Every cabinet is within `RADIUS_KM` **by road** of its cluster centre. Only where that is
   impossible may a cabinet join a neighbouring cluster within `RADIUS_KM` in a straight line; it is then flagged as
   a *straight-line fallback*.

Every cabinet ends up in exactly one cluster. A cluster that fits no team is broken up and its cabinets join the
nearest cluster as *leftovers*. Flagged cabinets (fallback and leftover) count for their cluster's demand, but they
don't move its centre and are drawn grey on the map.

**People available.** The demand is scaled so the clusters use about `TOTAL_PEOPLE` people: down when there is more
work than people, up (at most `MAX_DEMAND_SCALE` times) when there is less. Real tickets are always shown next to the
scaled ones.

**Steps**

| Step | What it does |
|---|---|
| Load and check | Read the cabinets of the partition and scale the demand to the people available. |
| Demand Layer | Decide how many clusters are needed and share the cabinets out evenly, none above `CAP`. |
| Distance Layer | Measure road distances; move, take out or split until every cabinet is within `RADIUS_KM` by road. |
| Minimum demand | Bring small clusters up to a full team's minimum: merge, share out or take over cabinets. |
| Smaller teams | Give the clusters that are still small a team of one person fewer each round. |
| Leftovers | Break up the clusters no team fits; their cabinets join the nearest cluster. |
| Save people | Pass cabinets on where that lets a cluster drop a person. |
| Fit the people available | Scale the demand again and rerun until the clusters fit `TOTAL_PEOPLE`. |
| Secondary cabinets | List nearby cabinets of other clusters as backup work. |
| Validation, outputs | Check every rule again, then write the tables and the map to `OUTPUT_FOLDER`. |

**Data.** `DATA_FILE` needs the columns `cabinetid, cabinetlat, cabinetlong, demand` (tickets per day); upper or lower
case doesn't matter. An optional `partition` column says whose work a cabinet is (`TRIS_1`, `TROS_1`, ...). One run
plans one partition; `PartitionPipeline.ipynb` plans them all and combines the results.

**Distances.** *Road* km come from OSRM, route planning on OpenStreetMap roads, without traffic. *Straight-line*
(haversine) km are used as a quick first filter and for the fallback.

## Setup and parameters
The next cell holds the files and the laptop settings. Paths are relative to the project root. The model rules are
not set here but in `parameters.csv` (next section). `PARTITION` picks the partition to plan; it can stay `None` for a
file without a partition column or with only one partition.

In [ ]:
DATA_FILE = "Dataset/mock_cabinets.csv"    # input file, relative to the project root
OUTPUT_FOLDER = "outputs"                # results, relative to the project root
PARAMETER_FILE = "parameters.csv"        # the model rules, one row per partition (see Model rules)
PARTITION = "TRIS_1"                     # plan only this partition's cabinets (None = the whole file: a file
                                         # without a partition column, or with only one partition)
WORKER_FILE = "Dataset/mock_workers.csv"   # only to count the people available when TOTAL_PEOPLE is blank in
                                           # PARAMETER_FILE: the workers of the partition (None = no limit)
CACHE_FILE = None                        # saved road distances (None = OUTPUT_FOLDER/cache/osrm_cache.json)

# Road distances (Distance Layer)
OSRM_URL = "http://127.0.0.1:5000"  # our own OSRM server on this laptop: start it first with start_osrm.bat
                                    # (public demo server, mock data only: "https://router.project-osrm.org")
OSRM_PROFILE = "driving"
OSRM_BATCH = 100        # max coordinates (centres + cabinets) per OSRM request
OSRM_PAUSE_S = 0        # pause before every request: 0 for our own server, 1.0 for the public demo server
ALLOW_FALLBACK = False  # True: if OSRM can't be reached, use straight-line x 1.4 instead (approximate!)

In [ ]:
# Install any missing library into the Python that is running this notebook (e.g. Anaconda),
# so "Run All" works on a fresh machine. Already-installed libraries are left alone.
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "folium": "folium",
            "shapely": "shapely", "requests": "requests"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Python", sys.version.split()[0], "- all required libraries available")

In [ ]:
import json
import math
import os
import time
from pathlib import Path

# One thread for k-means: avoids a scikit-learn memory-leak warning on Windows/Anaconda (MKL).
# Must be set before numpy/scikit-learn are imported; the data is small, so speed is unaffected.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import folium
import numpy as np
import pandas as pd
import requests
import shapely
from shapely.geometry import mapping
from sklearn.cluster import KMeans

ROOT = Path.cwd()   # the project root (or the run folder, when the dashboard runs a copy)
DATA_PATH = ROOT / DATA_FILE
OUT_DIR = ROOT / OUTPUT_FOLDER
OUT_DIR.mkdir(parents=True, exist_ok=True)
# saved road distances, so re-runs don't ask OSRM again
CACHE_PATH = ROOT / CACHE_FILE if CACHE_FILE else OUT_DIR / "cache" / "osrm_cache.json"

EARTH_RADIUS_KM = 6371.0088
TOL = 1e-6          # numerical tolerance for demand sums
START = time.time()
pd.set_option("display.width", 140)

## Model rules – parameters.csv
One row per **partition**, so every team can have its own rules; a new partition only needs a new row. The row `ALL`
is for a cabinet file without a partition column.

| Column | Meaning | Blank means |
|---|---|---|
| `PEOPLE_PER_CLUSTER` | biggest team | required |
| `MIN_PEOPLE` | smallest team (at least 1) | required |
| `DEMAND_PER_PERSON` | most tickets/day for one person | required |
| `MIN_DEMAND_PER_PERSON` | fewest tickets/day that make a worthwhile day for one person | required |
| `SOLO_MAX_DEMAND` | one person may take up to this when the cluster can't get down to `DEMAND_PER_PERSON` | no special case |
| `RADIUS_KM` | every cabinet within this road distance of its centre | required |
| `MAX_SECONDARY` | backup cabinets kept per cluster, nearest first | keep all |
| `TOTAL_PEOPLE` | people available per day | the partition's workers in `WORKER_FILE` (no limit if `WORKER_FILE = None`) |
| `MAX_DEMAND_SCALE` | most the demand may be scaled up | 3 |
| `FIT_ROUNDS` | most extra runs to fit `TOTAL_PEOPLE` | 5 |
| `SEED` | random seed for k-means | 67 |
| `BALANCE_TOL` | the Demand Layer stops evening out once the heaviest and lightest clusters are this close (tickets/day) | 1.0 |

Other columns, such as `notes`, are ignored. The notebook stops if the partition has no row (it prints one to copy
in), a required cell is empty or a number is out of range.

In [ ]:
RULES = ["PEOPLE_PER_CLUSTER", "MIN_PEOPLE", "DEMAND_PER_PERSON", "MIN_DEMAND_PER_PERSON", "SOLO_MAX_DEMAND",
         "RADIUS_KM", "MAX_SECONDARY", "TOTAL_PEOPLE", "MAX_DEMAND_SCALE", "FIT_ROUNDS", "SEED", "BALANCE_TOL"]
REQUIRED_RULES = ["PEOPLE_PER_CLUSTER", "MIN_PEOPLE", "DEMAND_PER_PERSON", "MIN_DEMAND_PER_PERSON", "RADIUS_KM"]
WHOLE_RULES = ["PEOPLE_PER_CLUSTER", "MIN_PEOPLE", "MAX_SECONDARY", "TOTAL_PEOPLE", "FIT_ROUNDS", "SEED"]
BLANK_MEANS = {"SOLO_MAX_DEMAND": None, "MAX_SECONDARY": None, "TOTAL_PEOPLE": None, "MAX_DEMAND_SCALE": 3,
               "FIT_ROUNDS": 5, "SEED": 67, "BALANCE_TOL": 1.0}     # SOLO_MAX_DEMAND None = DEMAND_PER_PERSON
EXAMPLE_ROW = [6, 1, 6, 4, 7, 15, 10, "", 3, 5, 67, 1.0]            # printed as a row to copy for a new partition


def column(df, name):
    # The column called `name` in any upper/lower case, or None.
    return next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)


def partition_of(path, partition):
    # The partition this run plans: PARTITION, or the file's only partition, or "ALL" for a file without a
    # partition column.
    df = pd.read_csv(path, dtype=str)
    col = column(df, "partition")
    if col is None:
        assert partition is None, f"{path.name} has no partition column, so PARTITION must be None"
        return "ALL"
    names = df[col].fillna("").str.strip()
    assert (names != "").all(), f"{path.name}: {int((names == '').sum())} cabinet(s) have no partition"
    found = sorted(names.unique())
    if partition is None:
        assert len(found) == 1, (f"{path.name} has {len(found)} partitions ({', '.join(found)}): set PARTITION to one "
                                 "of them, or run PartitionPipeline.ipynb to plan them all")
        return found[0]
    assert partition in found, f"{path.name} has no cabinets in partition {partition} (it has: {', '.join(found)})"
    return partition


def count_workers(path, partition):
    # Workers of this partition in the worker file (all of them for "ALL" or a file without a partition column).
    df = pd.read_csv(path, dtype=str)
    col = column(df, "partition")
    if col is None or partition == "ALL":
        return len(df)
    return int((df[col].fillna("").str.strip() == partition).sum())


def read_parameters(path, partition):
    # The rules of one partition from the parameters file: {rule: value}. Blank cells get BLANK_MEANS.
    table = pd.read_csv(path, dtype=str).fillna("")
    part = column(table, "partition")
    absent = ([] if part is not None else ["partition"]) + [r for r in RULES if column(table, r) is None]
    assert not absent, f"{path.name} has no column {', '.join(absent)}"
    names = table[part].str.strip()
    dup = names[names.duplicated()].unique()
    assert not len(dup), f"{path.name}: partition listed more than once: {', '.join(dup)}"
    if partition not in set(names):
        raise ValueError(f"{path.name} has no row for partition {partition}. Add one, for example with the usual "
                         "rules:\n" + ",".join(["partition", *RULES]) + "\n"
                         + ",".join(map(str, [partition, *EXAMPLE_ROW])))
    row = table[names == partition].iloc[0]
    rules = {}
    for r in RULES:
        text = row[column(table, r)].strip()
        if text == "":
            assert r not in REQUIRED_RULES, f"{path.name}, {partition}: {r} must be filled in"
            rules[r] = BLANK_MEANS[r]
            continue
        try:
            value = float(text)
        except ValueError:
            raise ValueError(f"{path.name}, {partition}: {r} = {text!r} is not a number") from None
        assert r not in WHOLE_RULES or value == int(value), f"{path.name}, {partition}: {r} must be a whole number"
        rules[r] = int(value) if value == int(value) else value
    if rules["SOLO_MAX_DEMAND"] is None:
        rules["SOLO_MAX_DEMAND"] = rules["DEMAND_PER_PERSON"]
    return rules


RUN_PARTITION = partition_of(DATA_PATH, PARTITION)
RULES_USED = read_parameters(ROOT / PARAMETER_FILE, RUN_PARTITION)
people_from = PARAMETER_FILE
if RULES_USED["TOTAL_PEOPLE"] is None and WORKER_FILE is not None:
    RULES_USED["TOTAL_PEOPLE"] = count_workers(ROOT / WORKER_FILE, RUN_PARTITION)
    people_from = f"the workers of {RUN_PARTITION} in {WORKER_FILE}"
    assert RULES_USED["TOTAL_PEOPLE"] > 0, (f"{WORKER_FILE} has no workers in partition {RUN_PARTITION}: add them, or "
                                            f"fill in TOTAL_PEOPLE for {RUN_PARTITION} in {PARAMETER_FILE}")

PEOPLE_PER_CLUSTER = RULES_USED["PEOPLE_PER_CLUSTER"]
MIN_PEOPLE = RULES_USED["MIN_PEOPLE"]
DEMAND_PER_PERSON = RULES_USED["DEMAND_PER_PERSON"]
MIN_DEMAND_PER_PERSON = RULES_USED["MIN_DEMAND_PER_PERSON"]
SOLO_MAX_DEMAND = RULES_USED["SOLO_MAX_DEMAND"]
RADIUS_KM = RULES_USED["RADIUS_KM"]
MAX_SECONDARY = RULES_USED["MAX_SECONDARY"]
TOTAL_PEOPLE = RULES_USED["TOTAL_PEOPLE"]
MAX_DEMAND_SCALE = RULES_USED["MAX_DEMAND_SCALE"]
FIT_ROUNDS = RULES_USED["FIT_ROUNDS"]
SEED = RULES_USED["SEED"]
BALANCE_TOL = RULES_USED["BALANCE_TOL"]
CAP = PEOPLE_PER_CLUSTER * DEMAND_PER_PERSON              # max tickets/day per cluster
MIN_DEMAND = PEOPLE_PER_CLUSTER * MIN_DEMAND_PER_PERSON   # min tickets/day for a full team

assert PEOPLE_PER_CLUSTER >= 1, "PEOPLE_PER_CLUSTER must be at least 1"
assert 1 <= MIN_PEOPLE <= PEOPLE_PER_CLUSTER, "MIN_PEOPLE must be between 1 and PEOPLE_PER_CLUSTER"
assert 0 < MIN_DEMAND_PER_PERSON <= DEMAND_PER_PERSON, "MIN_DEMAND_PER_PERSON must be between 0 and DEMAND_PER_PERSON"
assert SOLO_MAX_DEMAND >= DEMAND_PER_PERSON, "SOLO_MAX_DEMAND must be at least DEMAND_PER_PERSON (blank = the same)"
assert RADIUS_KM > 0, "RADIUS_KM must be above 0"
assert MAX_SECONDARY is None or MAX_SECONDARY >= 1, "MAX_SECONDARY must be at least 1 (blank = keep all)"
assert TOTAL_PEOPLE is None or TOTAL_PEOPLE >= 1, "TOTAL_PEOPLE must be at least 1 (blank = count the workers)"
assert MAX_DEMAND_SCALE >= 1, "MAX_DEMAND_SCALE must be at least 1 (1 = never scale up)"
assert FIT_ROUNDS >= 0, "FIT_ROUNDS must be 0 or more"
np.random.seed(SEED)

print(f"Partition {RUN_PARTITION}: rules from {PARAMETER_FILE}; TOTAL_PEOPLE from {people_from}"
      + (" (no limit)" if TOTAL_PEOPLE is None else ""))
pd.Series(RULES_USED, name=RUN_PARTITION, dtype=object).to_frame()


## Load and check the data
We read the partition's cabinets; the file itself is never changed. Cabinets with zero demand need no visits and are
skipped with a warning. A single cabinet above `CAP` could never fit in one day, so the notebook stops if it finds
one.

**Scaling to the people available.** `TOTAL_PEOPLE` people can take `TOTAL_PEOPLE` × `DEMAND_PER_PERSON` tickets/day.
Every cabinet's demand is multiplied by the same factor, `DEMAND_SCALE`, so the total matches. The busy/quiet pattern
between cabinets stays the same. Scaling up doesn't create work: it plans more, smaller clusters so everyone is used,
and each person's real day gets lighter. The real value is kept as `raw_demand`; all steps below use the scaled
`demand`.

Teams are rounded up per cluster, so the clusters usually need a few more people than this first scale allows. *Save
people* and *Fit the people available* close that gap.

**How many clusters.** At least `ceil(total demand / CAP)` (every cluster full), at most
`floor(total demand / SMALLEST_MIN)` (every cluster one person at the minimum).

In [ ]:
def read_cabinets(path, partition):
    # Read the cabinet file and give the four columns we use their usual names. Upper/lower case doesn't matter, so
    # "cabinetid" works as well as "CabinetId". Other columns (e.g. total_tickets, total_days) are not needed.
    # Only the cabinets of `partition` are kept ("ALL" = a file without a partition column: all of them).
    df = pd.read_csv(path)
    by_lower = {str(c).strip().lower(): c for c in df.columns}
    columns = ["CabinetId", "cabinetlat", "cabinetlong", "demand"]
    absent = [c for c in columns if c.lower() not in by_lower]
    assert not absent, (f"{path.name} has no column {', '.join(absent)}. "
                        "Needed: cabinetid (or CabinetId), cabinetlat, cabinetlong, demand.")
    if partition != "ALL":
        keep = (df[by_lower["partition"]].astype(str).str.strip() == partition).to_numpy()
        print(f"Partition {partition}: {int(keep.sum())} of {len(df)} cabinets in {path.name}")
        df = df[keep]
    return df.rename(columns={by_lower[c.lower()]: c for c in columns})[columns].reset_index(drop=True)


cabinets = read_cabinets(DATA_PATH, RUN_PARTITION)
# the same cabinet may be in two partitions (e.g. TRIS and TROS), but only once in each
assert cabinets["CabinetId"].is_unique, f"CabinetId must be unique within partition {RUN_PARTITION}"
no_demand = cabinets["demand"] <= 0
if no_demand.any():
    ids = cabinets.loc[no_demand, "CabinetId"].tolist()
    print(f"WARNING: skipping {len(ids)} cabinet(s) with zero demand: "
          + ", ".join(ids[:10]) + (" ..." if len(ids) > 10 else ""))
    cabinets = cabinets[~no_demand].reset_index(drop=True)

# Fit the demand to the people available: scale every cabinet by the same factor, down if the total is too big,
# up (at most MAX_DEMAND_SCALE) if it is too small
RAW_TOTAL_DEMAND = cabinets["demand"].sum()
PEOPLE_CAPACITY = None if TOTAL_PEOPLE is None else TOTAL_PEOPLE * DEMAND_PER_PERSON
DEMAND_SCALE = 1.0 if PEOPLE_CAPACITY is None else min(MAX_DEMAND_SCALE, PEOPLE_CAPACITY / RAW_TOTAL_DEMAND)
cabinets["raw_demand"] = cabinets["demand"]
cabinets["demand"] = cabinets["raw_demand"] * DEMAND_SCALE
if DEMAND_SCALE < 1:
    print(f"Demand scaled down to {DEMAND_SCALE:.2%} to fit {TOTAL_PEOPLE} people x {DEMAND_PER_PERSON} = "
          f"{PEOPLE_CAPACITY} tickets/day: {RAW_TOTAL_DEMAND:.2f} -> {cabinets['demand'].sum():.2f}")
elif DEMAND_SCALE > 1:
    print(f"Demand scaled up to {DEMAND_SCALE:.2%} to use {TOTAL_PEOPLE} people x {DEMAND_PER_PERSON} = "
          f"{PEOPLE_CAPACITY} tickets/day: {RAW_TOTAL_DEMAND:.2f} real -> {cabinets['demand'].sum():.2f} scaled "
          f"(real tickets per person are {1 / DEMAND_SCALE:.0%} of the scaled ones)"
          + (f"; capped at MAX_DEMAND_SCALE = {MAX_DEMAND_SCALE:g}" if DEMAND_SCALE == MAX_DEMAND_SCALE else ""))
elif TOTAL_PEOPLE is not None:
    print(f"Demand {RAW_TOTAL_DEMAND:.2f} tickets/day matches {TOTAL_PEOPLE} people exactly - not scaled")

for cab, d in zip(cabinets["CabinetId"], cabinets["demand"]):
    assert d <= CAP, f"cabinet {cab} has demand {d:.3f} above CAP – this data needs a hotspot step"

TOTAL_DEMAND = cabinets["demand"].sum()
SMALLEST_MIN = MIN_PEOPLE * MIN_DEMAND_PER_PERSON   # a cluster below this is too small even for the smallest team
K_MIN = math.ceil(TOTAL_DEMAND / CAP - 1e-9)
K_MAX = max(K_MIN, math.floor(TOTAL_DEMAND / SMALLEST_MIN + 1e-9))
print(f"{len(cabinets)} cabinets, total demand = {TOTAL_DEMAND:.3f} tickets/day "
      f"(busiest cabinet {cabinets['demand'].max():.3f})")
print(f"Clusters: at least {K_MIN} (all full teams at {CAP}/day), at most {K_MAX} "
      f"(all teams of {MIN_PEOPLE} at {SMALLEST_MIN}/day)")
cabinets.describe().round(3)

## Distance helpers
* **Haversine distance**: straight-line km between two points on the earth's surface.
* **Local km projection**: k-means and the outlines need flat x/y coordinates, so we flatten lat/long into km around
  the data's mean latitude. That is accurate for a city or a state, not for a whole country.
* **Cluster centre**: the plain average of its cabinets' latitude and longitude.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Great-circle distance in km; inputs broadcast like numpy arrays.
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


REF_LAT = cabinets["cabinetlat"].mean()
REF_LON = cabinets["cabinetlong"].mean()


def project_km(lat, lon):
    # Equirectangular projection to local km around the data's mean latitude.
    x = np.radians(np.asarray(lon) - REF_LON) * np.cos(np.radians(REF_LAT)) * EARTH_RADIUS_KM
    y = np.radians(np.asarray(lat) - REF_LAT) * EARTH_RADIUS_KM
    return np.column_stack([x, y])


def unproject_km(xy):
    # Inverse of project_km: local km (x, y) -> (long, lat) columns, the order map shapes use.
    lon = REF_LON + np.degrees(xy[:, 0] / (np.cos(np.radians(REF_LAT)) * EARTH_RADIUS_KM))
    lat = REF_LAT + np.degrees(xy[:, 1] / EARTH_RADIUS_KM)
    return np.column_stack([lon, lat])


# The cabinets as plain arrays, used by every step below (position i = row i of `cabinets`)
LAT = cabinets["cabinetlat"].to_numpy()
LON = cabinets["cabinetlong"].to_numpy()
DEM = cabinets["demand"].to_numpy(copy=True)
XY = project_km(LAT, LON)


def members_of(labels, p):
    # Positions of the cabinets in cluster p.
    return np.where(labels == p)[0]


def centre_of(members, flagged=None):
    # Centre (lat, long) of these cabinets: the simple average of their positions. Cabinets marked in `flagged`
    # (leftovers, and in the road version straight-line fallbacks) don't move a centre; if every cabinet is flagged,
    # all of them are used.
    members = np.asarray(members, dtype=int)
    if flagged is not None and (~flagged[members]).any():
        members = members[~flagged[members]]
    return LAT[members].mean(), LON[members].mean()


def cluster_stats(labels, flagged=None):
    # For every cluster: (cluster labels, each cabinet's index into them, demand, centre lat, centre long).
    # The centre leaves out the cabinets marked in `flagged` (see centre_of).
    parts, inv = np.unique(labels, return_inverse=True)
    w = np.ones(len(labels)) if flagged is None else (~flagged).astype(float)
    w = np.where(np.bincount(inv, w)[inv] > 0, w, 1.0)      # a cluster with only flagged cabinets uses them all
    n = np.bincount(inv, w)
    return parts, inv, np.bincount(inv, DEM), np.bincount(inv, LAT * w) / n, np.bincount(inv, LON * w) / n


def dist_to_centre(labels, flagged=None):
    # Straight-line km from every cabinet to its own cluster centre.
    parts, inv, load, clat, clon = cluster_stats(labels, flagged)
    return haversine_km(LAT, LON, clat[inv], clon[inv])


def fits_radius(members, radius_km, skip=None):
    # True if every cabinet in `members` is within radius_km of the members' centre. Cabinets marked in `skip`
    # (leftovers) have no distance limit and don't move the centre.
    members = np.asarray(members, dtype=int)
    if len(members) == 0:
        return True
    clat, clon = centre_of(members, skip)
    check = members if skip is None else members[~skip[members]]
    return bool((haversine_km(LAT[check], LON[check], clat, clon) <= radius_km + 1e-9).all())

## Cluster outlines and neighbouring cabinets
A cluster's **outline** is its convex hull: the rubber band around its cabinets. Two outlines overlap when they share
more than a border or a point. Right after k-means no outlines overlap, because k-means cuts the map into
straight-edged slices. After that, every change (a move, a merge, a split) is made only if the new outlines stay
clear.

**Neighbours.** Each cabinet's *area* is the part of the map closer to it than to any other cabinet (a Voronoi
diagram, cut off 1.5 km beyond the outermost cabinets). Two cabinets are neighbours when their areas share a border,
and a cabinet *touches* a cluster when one of its neighbours is in it. Moves are only tried into touching clusters,
which keeps the model fast on thousands of cabinets.

In [ ]:
AREA_MARGIN_KM = 1.5   # cabinet areas stop this far beyond the outermost cabinets
_OUTLINES = {}         # saved outlines, so unchanged clusters are not redrawn every time


def outline(members):
    # Rubber band (convex hull, in projected km) around the cabinets at these positions.
    members = np.sort(np.asarray(members, dtype=int))
    key = members.tobytes()
    if key not in _OUTLINES:
        if len(_OUTLINES) > 200_000:
            _OUTLINES.clear()
        _OUTLINES[key] = shapely.MultiPoint(XY[members]).convex_hull
    return _OUTLINES[key]


def overlap(a, b):
    # Two outlines overlap when they share more than a border or a single point.
    return a.intersects(b) and not a.touches(b)


def outlines_clear(changes, labels, skip=None):
    # Would a change keep every outline clear of every other? `changes` = {cluster label: its cabinet positions
    # after the change} for every cluster the change touches (an empty list for a cluster that disappears).
    # Cabinets marked in `skip` (leftovers) are left out of the outlines. Only clusters whose bounding box comes
    # near a new outline can overlap it, so only those outlines are drawn.
    new = []
    for m in changes.values():
        m = np.asarray(m, dtype=int)
        if skip is not None:
            m = m[~skip[m]]
        if len(m):
            new.append(outline(m))
    for k, shape in enumerate(new):
        if any(overlap(shape, other) for other in new[k + 1:]):
            return False
    keep = ~np.isin(labels, list(changes))
    if skip is not None:
        keep &= ~skip
    idx = np.where(keep)[0]
    if len(idx) == 0 or not new:
        return True
    order = idx[np.argsort(labels[idx], kind="stable")]       # the other clusters' cabinets, grouped by cluster
    starts = np.r_[0, np.flatnonzero(np.diff(labels[order])) + 1]
    ends = np.r_[starts[1:], len(order)]
    x, y = XY[order, 0], XY[order, 1]
    x0, x1 = np.minimum.reduceat(x, starts), np.maximum.reduceat(x, starts)
    y0, y1 = np.minimum.reduceat(y, starts), np.maximum.reduceat(y, starts)
    for shape in new:
        bx0, by0, bx1, by1 = shape.bounds
        for g in np.where((x0 <= bx1) & (x1 >= bx0) & (y0 <= by1) & (y1 >= by0))[0]:
            if overlap(shape, outline(order[starts[g]:ends[g]])):
                return False
    return True


def change_ok(changes, labels, radius_km, skip=None):
    # A change is allowed if every cluster it touches keeps all its cabinets within radius_km of its new centre
    # and no outlines overlap.
    return (all(fits_radius(m, radius_km, skip) for m in changes.values())
            and outlines_clear(changes, labels, skip))


def can_move(i, dest, labels, radius_km=None):
    # Moving cabinet i into cluster dest keeps the outlines clear (and, if radius_km is given, the distance rule).
    src = labels[i]
    rest = members_of(labels, src)
    changes = {src: rest[rest != i], dest: np.append(members_of(labels, dest), i)}
    return outlines_clear(changes, labels) if radius_km is None else change_ok(changes, labels, radius_km)


def cabinet_areas(margin_km):
    # Each cabinet's area (its Voronoi cell, in projected km), cut off margin_km beyond the outermost cabinets.
    border = shapely.MultiPoint(XY).convex_hull.buffer(margin_km)
    if len(XY) == 1:
        return [border]
    cells = [None] * len(XY)
    tree = shapely.STRtree(shapely.points(XY))
    for cell in shapely.voronoi_polygons(shapely.MultiPoint(XY), extend_to=border).geoms:
        for i in tree.query(cell, predicate="contains"):   # cabinets at the very same spot share one area
            cells[i] = cell.intersection(border)
    return cells


def neighbours_of(cells):
    # Two cabinets are neighbours when their areas share a border line (touching at one corner is not enough).
    tree = shapely.STRtree(cells)
    nbrs = [set() for _ in cells]
    for i, cell in enumerate(cells):
        for j in tree.query(cell, predicate="intersects"):
            if j != i and cell.intersection(cells[j]).length > 1e-9:
                nbrs[i].add(int(j))
    return nbrs


CABINET_CELLS = cabinet_areas(AREA_MARGIN_KM)
NEIGHBOURS = neighbours_of(CABINET_CELLS)
EDGES = np.array([(i, j) for i, js in enumerate(NEIGHBOURS) for j in js], dtype=int).reshape(-1, 2)


def touching_clusters(members, labels):
    # Clusters (other than the cabinets' own) that these cabinets touch.
    members = np.atleast_1d(members)
    return {labels[j] for i in members for j in NEIGHBOURS[i]} - set(labels[members])


def cluster_areas(cluster_of, skip=None):
    # One shape per cluster (in long/lat): the areas of its cabinets joined together (used for the map). Cabinets
    # marked in `skip` (flagged ones) are left out, so a cluster made only of them gets no shape.
    keep = np.ones(len(cluster_of), dtype=bool) if skip is None else ~skip
    return {c: shapely.transform(shapely.unary_union([CABINET_CELLS[i] for i in np.where((cluster_of == c) & keep)[0]]),
                                 unproject_km)
            for c in np.unique(cluster_of[keep])}


print(f"{len(cabinets)} cabinet areas, {len(EDGES) // 2} neighbour links "
      f"(on average {len(EDGES) / len(cabinets):.1f} neighbours per cabinet)")

## Demand Layer – split by demand
1. `k = ceil(total demand / CAP)`: the fewest clusters that could hold all the demand.
2. **Demand-weighted k-means** with `k` groups. Busy cabinets pull the centres towards them, which already evens out
   the demand.
3. **Repair**: k-means doesn't know `CAP`, so some clusters may be over it. We move cabinets from the most overloaded
   cluster into touching clusters that have room, the lightest one within `RADIUS_KM` first. If no neighbour has room
   for a whole cabinet, we pass one on to a neighbour with a little room when that lowers the total overload; the
   overload then moves on further, like water finding a way out.
4. If the repair gets stuck, the most overloaded cluster is **split in two** (k-means with k = 2) and the repair
   carries on. A split cuts along a straight line, so outlines stay clear.
5. **Balancing**: we move single border cabinets from heavier to lighter touching clusters while that brings the two
   closer, until the heaviest and lightest are within `BALANCE_TOL` of each other or no move is left.

Distances here are straight-line, as a cheap first filter: a road is never shorter, so a cabinet that fails here
would fail by road too. The road rule comes next.

In [ ]:
MAX_REPAIR_MOVES = 500
MAX_BALANCE_MOVES = max(500, len(cabinets))


def repair_overloads(labels, cap, radius_km, max_moves=MAX_REPAIR_MOVES):
    # Move cabinets out of overloaded clusters into touching clusters. Returns (labels, n_moves, success).
    labels = labels.copy()
    for move in range(max_moves + 1):
        parts, inv, load_arr, clat, clon = cluster_stats(labels)
        load = dict(zip(parts, load_arr))
        over = [p for p in parts if load[p] > cap + 1e-9]
        if not over:
            return labels, move, True
        if move == max_moves:
            break
        src = max(over, key=lambda p: load[p])
        centre = {p: (la, lo) for p, la, lo in zip(parts, clat, clon)}

        def km(i, q):
            return haversine_km(LAT[i], LON[i], *centre[q])

        options = [(i, q) for i in members_of(labels, src) for q in touching_clusters(i, labels)]
        fit = [(i, q) for i, q in options if load[q] + DEM[i] <= cap + 1e-9]
        dest = None
        # cabinets in order of how close they are to a cluster that has room for them
        for i in sorted({i for i, _ in fit}, key=lambda i: min(km(i, q) for j, q in fit if j == i)):
            targets = [q for j, q in fit if j == i]
            close = sorted((q for q in targets if km(i, q) <= radius_km), key=lambda q: load[q])
            far = sorted((q for q in targets if km(i, q) > radius_km), key=lambda q: km(i, q))
            dest = next((q for q in close + far if can_move(i, q, labels)), None)
            if dest is not None:
                labels[i] = dest
                break
        if dest is not None:
            continue
        # No touching cluster can take a whole cabinet: pass one on to a touching cluster with a little room
        # anyway, if that lowers the total overload (the overload then moves on from there).
        passes = []
        for i, q in options:
            before = (load[src] - cap) + max(0, load[q] - cap)
            after = max(0, load[src] - DEM[i] - cap) + max(0, load[q] + DEM[i] - cap)
            if after < before - 1e-9:
                passes.append((after - before, km(i, q), i, q))
        choice = next(((i, q) for _, _, i, q in sorted(passes) if can_move(i, q, labels)), None)
        if choice is None:
            break   # nowhere to put anything: caller will increase k
        labels[choice[0]] = choice[1]
    return labels, move, False


def load_spread(labels):
    # Heaviest minus lightest cluster demand.
    load = cluster_stats(labels)[2]
    return load.max() - load.min()


def max_radius(members):
    # Largest straight-line distance (km) from one of these cabinets to their centre.
    return haversine_km(LAT[members], LON[members], LAT[members].mean(), LON[members].mean()).max()


def balance_loads(labels, cap, radius_km, tol, max_moves=MAX_BALANCE_MOVES):
    # Even out cluster demand with single-cabinet moves from heavier to lighter touching clusters until
    # heaviest - lightest <= tol. Each accepted move has dem[i] < load[src] - load[dst], which strictly
    # lowers the sum of squared deviations from the mean demand, so the loop always ends.
    # Returns (labels, n_moves).
    labels = labels.copy()
    for move in range(max_moves):
        parts, inv, load, clat, clon = cluster_stats(labels)
        if load.max() - load.min() <= tol:
            return labels, move
        # every (cabinet, touching cluster) pair, from the neighbour links (each pair coded as one number)
        code = np.unique(EDGES[:, 0] * len(parts) + inv[EDGES[:, 1]])
        i, q = code // len(parts), code % len(parts)
        own = inv[i]
        keep = (q != own) & (DEM[i] < load[own] - load[q] - 1e-9) & (load[q] + DEM[i] <= cap + 1e-9)
        i, q, own = i[keep], q[keep], own[keep]
        extra = (haversine_km(LAT[i], LON[i], clat[q], clon[q])          # extra km vs. staying
                 - haversine_km(LAT[i], LON[i], clat[own], clon[own]))
        moved = False
        for k in np.argsort(extra, kind="stable"):
            ci, src, dst = i[k], parts[own[k]], parts[q[k]]
            before = members_of(labels, src)
            rest, after = before[before != ci], np.append(members_of(labels, dst), ci)
            if len(rest) == 0 or max_radius(after) > radius_km:
                continue
            if max_radius(rest) > max(radius_km, max_radius(before)):
                continue
            if not outlines_clear({src: rest, dst: after}, labels):
                continue
            labels[ci] = dst
            moved = True
            break
        if not moved:
            return labels, move
    return labels, max_moves


def demand_layer(cap, radius_km, seed, balance_tol):
    # Returns (labels, summary): a cluster label for every cabinet, and what the repair and balancing loops did.
    k = min(math.ceil(DEM.sum() / cap - 1e-9), len(DEM))
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(XY, sample_weight=DEM)
    labels, moves, ok = repair_overloads(labels, cap, radius_km)
    splits = 0
    while not ok:
        # the repair loop is stuck: split the most overloaded cluster in two (its halves have room for the
        # neighbours' overload too) and carry on repairing
        parts, inv, load, clat, clon = cluster_stats(labels)
        members = members_of(labels, parts[np.argmax(load)])
        halves = KMeans(n_clusters=2, n_init=10, random_state=seed).fit_predict(XY[members])
        labels[members[halves == 1]] = labels.max() + 1
        splits += 1
        labels, more, ok = repair_overloads(labels, cap, radius_km)
        moves += more
    summary = {"k_start": k, "repair_moves": moves, "overloaded_clusters_split": splits,
               "clusters_after_repair": len(np.unique(labels)), "spread_before": load_spread(labels)}
    labels, summary["balance_moves"] = balance_loads(labels, cap, radius_km, balance_tol)
    summary["spread_after"] = load_spread(labels)
    return labels, pd.Series(summary, name="value").to_frame()

In [ ]:
demand_labels, demand_log = demand_layer(CAP, RADIUS_KM, SEED, BALANCE_TOL)
demand_loads = cluster_stats(demand_labels)[2]
print(f"Demand Layer: {len(demand_loads)} clusters (possible range {K_MIN} to {K_MAX}), "
      f"demand per cluster {demand_loads.min():.2f} to {demand_loads.max():.2f} tickets/day")
print(f"({time.time() - START:.0f} s so far) spread = heaviest minus lightest cluster (tickets/day), "
      "before and after balancing")
demand_log.round(2)

## Distance Layer – road distance from each centre
Now rule 2: every cabinet within `RADIUS_KM` **by road** of its centre.

* **Road distance**: OSRM's table service gives the km from each centre to its cabinets in one request. A centre
  that doesn't sit on a road is first *snapped* to the nearest road; we warn if that is more than 1 km.
* **Server**: requests are batched (`OSRM_BATCH` points each) and retried on errors. If the server can't be reached
  the notebook stops. Only with `ALLOW_FALLBACK = True` does it use straight-line × 1.4 instead, with a warning that
  the results are approximate.
* **Cache**: every road distance is saved in the cache file (`CACHE_FILE`, or `OUTPUT_FOLDER/cache/`), so reruns
  don't ask OSRM again.
* **Shortcut**: a cabinet more than `RADIUS_KM` away in a straight line fails without asking OSRM.

We then take the cabinet farthest by road, while any is beyond `RADIUS_KM`, and:

1. **move** it to the lightest touching cluster that has room and keeps every cabinet of both clusters within reach;
2. otherwise **take it out on its own**, if the rest of its cluster is fine without it (the minimum demand step finds
   it a home later);
3. otherwise **split** its cluster in two.

After every change the centres and distances are worked out again. A cabinet never moves into the same cluster twice,
so the loop can't go back and forth.

In [ ]:
OSRM_RETRIES = 4         # attempts per request before giving up
FALLBACK_FACTOR = 1.4    # only used when ALLOW_FALLBACK = True
SNAP_WARN_KM = 1.0       # warn when a cluster centre is more than this from the nearest road
CACHE_SAVE_EVERY_S = 30  # while a step runs, the cache file is written at most this often


def load_cache(path):
    # Road distances saved by earlier runs: {"road_km": {...}, "snap_km": {...}}.
    if path.exists():
        return json.loads(path.read_text())
    return {"road_km": {}, "snap_km": {}}


def save_cache(cache, path, attempts=6):
    # Write to a temporary file, then swap it in. A synced folder (e.g. OneDrive) can lock the file for a
    # moment while uploading it, so the swap is retried a few times before giving up.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(cache))
    for attempt in range(attempts):
        try:
            tmp.replace(path)
            return
        except PermissionError:
            if attempt == attempts - 1:
                raise
            time.sleep(0.5 * 2 ** attempt)   # 0.5, 1, 2, 4, 8 s


def save_cache_if_due(force=False):
    # Writing the whole cache file takes a moment, so while a step runs it is done at most every
    # CACHE_SAVE_EVERY_S seconds. Every step that asks for road distances ends with force=True.
    if CACHE_STATE["unsaved"] and (force or time.time() - CACHE_STATE["saved_at"] >= CACHE_SAVE_EVERY_S):
        save_cache(CACHE, CACHE_PATH)
        CACHE_STATE.update(unsaved=False, saved_at=time.time())


CACHE = load_cache(CACHE_PATH)
CACHE_STATE = {"unsaved": False, "saved_at": time.time()}
OSRM_STATS = {"requests": 0, "fetched": 0, "cache_hits": 0, "shortcut": 0, "fallback": 0}
FALLBACK_KM = {}   # approximate distances from the fallback; never saved to the cache


def point_key(lat, lon):
    # "lat,lon" rounded to 5 decimals (about 1 m): used both in requests and as cache keys.
    return f"{lat:.5f},{lon:.5f}"


def key_point(key):
    lat, lon = map(float, key.split(","))
    return lat, lon


def pair_key(src, dst):
    return f"{OSRM_PROFILE}|{src}|{dst}"


def snap_km(lat, lon):
    # How far OSRM moved this point to reach the nearest road (km); NaN if never sent as a source.
    return CACHE["snap_km"].get(f"{OSRM_PROFILE}|{point_key(lat, lon)}", np.nan)


def make_batches(pairs, max_points):
    # Group (source, destination) pairs into requests that use at most max_points coordinates each.
    batches, srcs, dsts = [], [], []
    for s, d in pairs:
        extra = (s not in srcs) + (d not in dsts)
        if srcs and len(srcs) + len(dsts) + extra > max_points:
            batches.append((srcs, dsts))
            srcs, dsts = [], []
        if s not in srcs:
            srcs.append(s)
        if d not in dsts:
            dsts.append(d)
    if srcs:
        batches.append((srcs, dsts))
    return batches


def fetch_table(srcs, dsts):
    # One OSRM Table request: road distances from every point in srcs to every point in dsts.
    coords = ";".join(f"{lon},{lat}" for lat, lon in map(key_point, srcs + dsts))   # OSRM wants lon,lat
    url = (f"{OSRM_URL}/table/v1/{OSRM_PROFILE}/{coords}"
           f"?sources={';'.join(str(i) for i in range(len(srcs)))}"
           f"&destinations={';'.join(str(i) for i in range(len(srcs), len(srcs) + len(dsts)))}"
           f"&annotations=distance")
    problem = ""
    for attempt in range(1, OSRM_RETRIES + 1):
        time.sleep(OSRM_PAUSE_S if attempt == 1 else 5 * 2 ** (attempt - 2))   # pause, then 5, 10, 20 s
        try:
            r = requests.get(url, timeout=60)
            answer = r.json() if "json" in r.headers.get("content-type", "") else {}
            if r.ok and answer.get("code") == "Ok":
                OSRM_STATS["requests"] += 1
                return answer
            problem = f"HTTP {r.status_code} {answer.get('code', '')} {answer.get('message', r.text[:200])}"
            if r.status_code == 400:
                break   # the request itself is wrong (e.g. too many points): retrying won't help
        except requests.Timeout:
            problem = "no answer within 60 s"
        except requests.ConnectionError:
            problem = "cannot connect to the server"
        except (requests.RequestException, ValueError) as err:
            problem = type(err).__name__
        print(f"  OSRM attempt {attempt}/{OSRM_RETRIES} failed: {problem}")
    raise ConnectionError(f"Could not get road distances from OSRM at {OSRM_URL} ({problem.strip()}). "
                          "If this is our own server, start it with start_osrm.bat; otherwise check the "
                          "internet connection and OSRM_URL. Or set ALLOW_FALLBACK = True to use approximate "
                          "straight-line distances.")


def use_fallback(srcs, dsts, err):
    # Approximate road km = straight-line x FALLBACK_FACTOR. Only reached when ALLOW_FALLBACK = True.
    if OSRM_STATS["fallback"] == 0:
        print("!" * 100 + "\nWARNING: OSRM could not be reached, so road distances are APPROXIMATE "
              f"(straight-line x {FALLBACK_FACTOR}).\nThe results below do NOT check the real road rule. "
              f"Reason: {err}\n" + "!" * 100)
    for s in srcs:
        for d in dsts:
            FALLBACK_KM[(s, d)] = FALLBACK_FACTOR * float(haversine_km(*key_point(s), *key_point(d)))
    OSRM_STATS["fallback"] += len(srcs) * len(dsts)


def road_km(pairs):
    # Road km for each (source, destination) pair; each point is a (lat, lon) tuple. Distances already in
    # the cache are reused; only the missing ones are sent to OSRM, in batches. inf = no road route.
    keys = [(point_key(*s), point_key(*d)) for s, d in pairs]
    known = CACHE["road_km"]
    missing = sorted({k for k in keys if pair_key(*k) not in known and k not in FALLBACK_KM})
    OSRM_STATS["cache_hits"] += sum(pair_key(*k) in known for k in keys)
    fetched = False
    for srcs, dsts in make_batches(missing, OSRM_BATCH):
        if OSRM_STATS["fallback"]:   # OSRM already failed in this run: don't keep asking it
            use_fallback(srcs, dsts, None)
            continue
        try:
            answer = fetch_table(srcs, dsts)
        except ConnectionError as err:
            if not ALLOW_FALLBACK:
                raise
            use_fallback(srcs, dsts, err)
            continue
        for s, row, snapped in zip(srcs, answer["distances"], answer["sources"]):
            CACHE["snap_km"][f"{OSRM_PROFILE}|{s}"] = snapped["distance"] / 1000
            for d, metres in zip(dsts, row):
                known[pair_key(s, d)] = None if metres is None else metres / 1000
        OSRM_STATS["fetched"] += len(srcs) * len(dsts)
        fetched = True
    if fetched:
        CACHE_STATE["unsaved"] = True
        save_cache_if_due()
    km = [known[pair_key(*k)] if pair_key(*k) in known else FALLBACK_KM[k] for k in keys]
    return np.array([np.inf if x is None else x for x in km], dtype=float)

In [ ]:
MAX_DISTANCE_ROUNDS = max(300, len(cabinets))


def road_to_centre(labels, radius_km, flagged=None):
    # Road and straight-line km from every cabinet to its own cluster centre (flagged cabinets don't move a centre).
    # Shortcut: a cabinet more than radius_km away in a straight line gets road = inf (fails) without asking OSRM.
    parts, inv, load, clat, clon = cluster_stats(labels, flagged)
    straight = haversine_km(LAT, LON, clat[inv], clon[inv])
    road = np.full(len(LAT), np.inf)
    ask = np.where(straight <= radius_km)[0]
    OSRM_STATS["shortcut"] += len(LAT) - len(ask)
    road[ask] = road_km([((clat[inv[i]], clon[inv[i]]), (LAT[i], LON[i])) for i in ask])
    return road, straight


def groups_pass(groups, fallback, radius_km, exempt=None):
    # For each candidate cluster (cabinet positions): does every cabinet pass from the group's own centre?
    # Normal cabinets need road <= radius_km; straight-line-fallback cabinets need straight line <= radius_km;
    # exempt cabinets (leftovers) are not checked. The centre is worked out from the normal cabinets only: fallback
    # and exempt cabinets don't move it. The road distances for all groups are fetched together (straight-line
    # shortcut first).
    ok, pairs, owner = [True] * len(groups), [], []
    flagged = fallback if exempt is None else fallback | exempt
    for g, members in enumerate(groups):
        members = np.asarray(members, dtype=int)
        if len(members) == 0:
            continue
        clat, clon = centre_of(members, flagged)
        checked = members if exempt is None else members[~exempt[members]]
        if (haversine_km(LAT[checked], LON[checked], clat, clon) > radius_km + 1e-9).any():
            ok[g] = False
            continue
        for i in checked[~fallback[checked]]:
            pairs.append(((clat, clon), (LAT[i], LON[i])))
            owner.append(g)
    for g, km in zip(owner, road_km(pairs)):
        ok[g] = ok[g] and km <= radius_km + 1e-9
    return ok


def first_passing(options, changes_of, labels, fallback, radius_km, exempt=None, chunk=4):
    # The first option (in the given order) whose change keeps the outlines clear and passes the distance rule;
    # None if none does. changes_of(option) = {cluster label: its cabinet positions after the change}.
    # The outline check needs no OSRM, so it goes first; the remaining options are checked by road a few at a time.
    batch = []
    for option in list(options) + [None]:
        if option is not None:
            if outlines_clear(changes_of(option), labels):
                batch.append(option)
            if len(batch) < chunk:
                continue
        if batch:
            groups = [list(changes_of(b).values()) for b in batch]
            ok = groups_pass([g for gs in groups for g in gs], fallback, radius_km, exempt)
            pos = 0
            for b, gs in zip(batch, groups):
                if all(ok[pos:pos + len(gs)]):
                    return b
                pos += len(gs)
            batch = []
    return None


def best_new_cluster(i, labels, cap, radius_km, tried, fallback):
    # The lightest touching cluster that can take cabinet i without going over cap, with every cabinet of both
    # clusters passing the road rule and the outlines clear. None if no cluster qualifies.
    load = {p: DEM[labels == p].sum() for p in touching_clusters(i, labels)}
    src = labels[i]
    rest = members_of(labels, src)
    rest = rest[rest != i]
    options = sorted((q for q in load if (i, q) not in tried and load[q] + DEM[i] <= cap + 1e-9), key=load.get)
    return first_passing(options, lambda q: {src: rest, q: np.append(members_of(labels, q), i)},
                         labels, fallback, radius_km)


def distance_layer(labels, cap, radius_km, seed, max_rounds=MAX_DISTANCE_ROUNDS):
    # Fix every cabinet that is too far by road from its cluster centre, worst first: move it, take it out on its
    # own, or split its cluster in two. Returns (labels, action log, number of cabinets too far at first).
    labels = labels.copy()
    no_fallback = np.zeros(len(DEM), dtype=bool)
    tried, actions, first_fail = set(), [], None
    for rnd in range(max_rounds + 1):
        road, straight = road_to_centre(labels, radius_km)
        fail = np.where(road > radius_km + 1e-9)[0]
        if first_fail is None:
            first_fail = len(fail)
        if len(fail) == 0:
            return labels, pd.DataFrame(actions), first_fail
        if rnd == max_rounds:
            break
        i = fail[np.argmax(road[fail])]
        entry = {"round": rnd + 1, "CabinetId": cabinets["CabinetId"].iloc[i],
                 "road_km": road[i], "straight_km": straight[i]}
        dest = best_new_cluster(i, labels, cap, radius_km, tried, no_fallback)
        if dest is not None:
            labels[i] = dest
            tried.add((i, dest))
            entry["action"] = "moved to another cluster"
        else:
            members = members_of(labels, labels[i])
            if len(members) < 2:
                raise RuntimeError(f"cabinet {entry['CabinetId']}: OSRM finds no road route to its own location")
            rest, new = members[members != i], labels.max() + 1
            if first_passing([0], lambda _: {labels[i]: rest, new: [i]}, labels, no_fallback, radius_km) is not None:
                labels[i] = new
                entry["action"] = "taken out on its own"
            else:
                halves = KMeans(n_clusters=2, n_init=10, random_state=seed).fit_predict(XY[members])
                labels[members[halves == 1]] = new
                entry["action"] = "cluster split in two"
        actions.append(entry)
    raise RuntimeError(f"Distance Layer did not settle after {max_rounds} rounds: {len(fail)} cabinet(s) are "
                       f"still more than {radius_km} km by road from their cluster centre")

In [ ]:
labels, distance_log, first_fail = distance_layer(demand_labels, CAP, RADIUS_KM, SEED)
save_cache_if_due(force=True)   # write the new road distances to the cache file
counts = distance_log["action"].value_counts() if len(distance_log) else pd.Series(dtype=int)
print(f"Road check at first: {first_fail} of {len(cabinets)} cabinets were more than {RADIUS_KM} km "
      f"by road from their cluster centre")
print(f"Fixed with {counts.get('moved to another cluster', 0)} move(s), "
      f"{counts.get('taken out on its own', 0)} cabinet(s) taken out on their own and "
      f"{counts.get('cluster split in two', 0)} split(s): {len(np.unique(labels))} clusters now")
print(f"OSRM: {OSRM_STATS['requests']} requests ({OSRM_STATS['fetched']} distances fetched), "
      f"{OSRM_STATS['cache_hits']} distances taken from the cache, "
      f"{OSRM_STATS['shortcut']} cabinet checks failed by the straight-line shortcut (no OSRM call)")
if OSRM_STATS["fallback"]:
    print(f"WARNING: {OSRM_STATS['fallback']} distances are APPROXIMATE (straight-line x {FALLBACK_FACTOR})")
distance_log.round(2)

## Minimum demand – bring small clusters up to a full team
A cluster below `MIN_DEMAND` (a full team at its minimum) is not a worthwhile day for that team. We take them one at a
time, lightest first, and try:

1. **merge** with a touching cluster, if together they stay within `CAP`;
2. **share out** its cabinets to touching clusters that have room (only if every cabinet finds a place);
3. **top up** by taking over cabinets from touching clusters that stay at their own minimum.

These keep the road rule. Only a cluster too small for any team (below `SMALLEST_MIN`) may then use the
**straight-line fallback**: join the nearest touching cluster, or share its cabinets out, with the straight-line
rule; cabinets beyond `RADIUS_KM` by road are flagged. A bigger cluster is never pulled in by straight line: if 1 to 3
fail, it waits for a smaller team in the next step, so a quiet town keeps its own cluster and centre.

A cluster where nothing works is tried again whenever something else changes. Every change removes a cluster or makes
a small one bigger, so the step always ends.

In [ ]:
MAX_MIN_ROUNDS = max(2000, 10 * len(cabinets))   # a safety cap only; real runs need about 2 per cabinet


def min_demand_step(labels, cap, min_demand, radius_km, active=None, fallback=None, max_rounds=MAX_MIN_ROUNDS,
                    fallback_below=None):
    # Bring every cluster below min_demand up to it, lightest first: merge, share out, top up (road rule), then the
    # straight-line fallback. `active` = the clusters this step may change (None = all of them); `fallback` = the
    # straight-line fallback flags so far (None = none yet); `fallback_below` = only clusters below this demand may
    # use the straight-line fallback (None = all of them).
    # Returns (labels, straight-line fallback flag per cabinet, action log, labels of clusters left below min_demand).
    labels = labels.copy()
    fallback = np.zeros(len(DEM), dtype=bool) if fallback is None else fallback.copy()
    active = None if active is None else {int(p) for p in active}

    def load(p):
        return DEM[labels == p].sum()

    def members(p):
        return members_of(labels, p)

    def km_to_centre(i, p):
        # straight-line km from cabinet i to the centre of cluster p (fallback cabinets don't move a centre)
        return haversine_km(LAT[i], LON[i], *centre_of(members(p), fallback))

    def centre_gap(a, b):
        # straight-line km between the centres of clusters a and b
        return haversine_km(*centre_of(members(a), fallback), *centre_of(members(b), fallback))

    def flag_far(t, moved):
        # After moving cabinets into t by straight line: the ones too far by road from t's centre (worked out without
        # them) are flagged as fallback. If counting the others in the centre would break the road rule for t, they
        # are all flagged. Returns the number of moved cabinets flagged.
        moved = np.atleast_1d(moved)
        without = fallback.copy()
        without[moved] = True
        centre = centre_of(members(t), without)
        far = road_km([(centre, (LAT[i], LON[i])) for i in moved]) > radius_km
        fallback[moved[far]] = True
        if not far.all() and not groups_pass([members(t)], fallback, radius_km)[0]:
            fallback[moved] = True
        return int(fallback[moved].sum())

    def check(options, changes_of, fb=None):
        return first_passing(options, changes_of, labels, fallback if fb is None else fb, radius_km)

    def touching(m):
        # the clusters these cabinets touch that this step may change
        found = touching_clusters(m, labels)
        return found if active is None else found & active

    def merge(s):
        opts = sorted((t for t in touching(members(s)) if load(s) + load(t) <= cap + 1e-9),
                      key=lambda t: (load(s) + load(t) < min_demand - 1e-9, centre_gap(s, t)))
        t = check(opts, lambda t: {s: [], t: np.concatenate([members(s), members(t)])})
        if t is None:
            return None
        labels[labels == s] = t
        return "merged with a neighbour"

    def share_out(s, fb=None):
        before, before_fallback, flagged = labels.copy(), fallback.copy(), 0
        while (labels == s).any():
            ms = members(s)
            opts = sorted(((i, t) for i in ms for t in touching(i)
                           if load(t) + DEM[i] <= cap + 1e-9),
                          key=lambda it: km_to_centre(*it))
            relaxed = None
            if fb:   # straight-line share-out: the cabinet being moved only needs to pass in a straight line
                relaxed = fallback.copy()
                relaxed[ms] = True
            choice = check(opts, lambda it: {s: ms[ms != it[0]], it[1]: np.append(members(it[1]), it[0])}, relaxed)
            if choice is None:
                labels[:], fallback[:] = before, before_fallback   # not every cabinet found a place: undo
                return None
            i, t = choice
            labels[i] = t
            if fb:
                flagged += flag_far(t, i)
        return (f"straight-line share-out to nearest neighbours ({flagged} cabinet(s) flagged)" if fb
                else "shared out to neighbours")

    def top_up(s):
        taken = 0
        while load(s) < min_demand - 1e-9:
            ms = members(s)
            opts = sorted(((i, t) for t in touching(ms) for i in members(t)
                           if load(t) - DEM[i] >= min_demand - 1e-9 and load(s) + DEM[i] <= cap + 1e-9
                           and s in touching_clusters(i, labels)),
                          key=lambda it: km_to_centre(it[0], s))
            choice = check(opts, lambda it: {s: np.append(ms, it[0]), it[1]: members(it[1])[members(it[1]) != it[0]]})
            if choice is None:
                break
            labels[choice[0]] = s
            taken += 1
        return f"took over {taken} cabinet(s)" if taken else None

    def straight_line_fallback(s):
        opts = sorted((t for t in touching(members(s)) if load(s) + load(t) <= cap + 1e-9),
                      key=lambda t: centre_gap(s, t))
        relaxed = fallback.copy()
        relaxed[labels == s] = True   # s's cabinets only need to pass in a straight line
        t = check(opts, lambda t: {s: [], t: np.concatenate([members(s), members(t)])}, relaxed)
        if t is None:
            return None
        moved = members(s)
        labels[moved] = t
        return f"straight-line fallback into nearest neighbour ({flag_far(t, moved)} cabinet(s) flagged)"

    log, stuck = [], set()
    for _ in range(max_rounds):
        parts, inv, load_arr, clat, clon = cluster_stats(labels)
        small = [p for p, d in zip(parts, load_arr) if d < min_demand - 1e-9 and p not in stuck
                 and (active is None or p in active)]
        if not small:
            return labels, fallback, pd.DataFrame(log), sorted(stuck)
        s = min(small, key=load)
        ms = members(s)
        entry = {"n_cabinets": len(ms), "demand_before": load(s),
                 "cabinets": ", ".join(cabinets["CabinetId"].iloc[ms[:3]]) + (" ..." if len(ms) > 3 else "")}
        action = merge(s) or share_out(s) or top_up(s)
        if action is None and (fallback_below is None or load(s) < fallback_below - 1e-9):
            action = straight_line_fallback(s) or share_out(s, fb=True)
        if action is None:
            stuck.add(s)
            action = "stuck for now"
        else:
            stuck.clear()   # something changed: clusters that were stuck get another try
        entry["action"] = action
        log.append(entry)
    raise RuntimeError(f"Minimum demand step did not settle after {max_rounds} rounds")

In [ ]:
labels, fallback, min_log, still_low = min_demand_step(labels, CAP, MIN_DEMAND, RADIUS_KM, fallback_below=SMALLEST_MIN)
save_cache_if_due(force=True)   # write the new road distances to the cache file
kinds = min_log["action"].str.split(" ").str[0].value_counts() if len(min_log) else pd.Series(dtype=int)
print(f"{len(min_log)} small-cluster step(s): {kinds.get('merged', 0)} merged, {kinds.get('shared', 0)} shared out, "
      f"{kinds.get('took', 0)} topped up, {kinds.get('straight-line', 0)} by straight-line fallback, "
      f"{kinds.get('stuck', 0)} 'stuck for now'; {len(np.unique(labels))} clusters now")
if fallback.any():
    print(f"Straight-line fallback cabinets: {', '.join(cabinets['CabinetId'][fallback])}")
if still_low:
    print(f"{len(still_low)} cluster(s) could not reach {MIN_DEMAND}/day: they are tried with a smaller team below")
print(f"OSRM so far: {OSRM_STATS['requests']} requests, {OSRM_STATS['cache_hits']} distances from the cache "
      f"({time.time() - START:.0f} s so far)")
min_log.round(2)

## Smaller teams – one person fewer each round
The clusters still below `MIN_DEMAND` sit in quiet areas. Instead of breaking them up we try smaller teams, from
`PEOPLE_PER_CLUSTER - 1` down to `MIN_PEOPLE`. In each round these clusters get one more minimum demand pass with the
smaller team's numbers, among themselves only, and every one that now fits the team gets it.

What can be left after the last round:

* **A little too much for one person** (between `DEMAND_PER_PERSON` and `SOLO_MAX_DEMAND`): it passes cabinets on
  if it can; otherwise one person takes it as the *special case* (dashed ring on the map).
* **Between two bands** (too much for one person, too little for two): it passes cabinets on, or takes some over
  until two people fit. A neighbour only gives or takes a cabinet if its own team stays the same.
* **Too small for any team**: broken up in the next step.

At the end every cluster has the team its demand calls for, and the three rules still hold.

In [ ]:
def team_band(people):
    # (minimum, maximum) tickets/day for a team of this many people.
    return people * MIN_DEMAND_PER_PERSON, people * DEMAND_PER_PERSON


def team_for(demand):
    # (team size, special case?) for this demand: the smallest team whose band holds it. If none does, one person
    # may still take up to SOLO_MAX_DEMAND as the special case. (None, False) if no team fits: below the smallest
    # minimum, or in a gap between two bands (7 to 8 tickets/day: too much for one person, too little for two).
    for team in range(MIN_PEOPLE, PEOPLE_PER_CLUSTER + 1):
        lo, hi = team_band(team)
        if lo - 1e-9 <= demand <= hi + 1e-9:
            return team, False
    if MIN_PEOPLE == 1 and team_band(1)[0] - 1e-9 <= demand <= SOLO_MAX_DEMAND + 1e-9:
        return 1, True
    return None, False


def fits_band(demand):
    # Some team fits this demand without the one-person special case.
    team, special = team_for(demand)
    return team is not None and not special


def smallest_team(demand):
    # The team for this demand. A demand no team fits (only a cluster of leftovers that no cluster had room for)
    # gets the fewest people who can carry it.
    team = team_for(demand)[0]
    return team if team is not None else min(PEOPLE_PER_CLUSTER,
                                             max(MIN_PEOPLE, math.ceil(demand / DEMAND_PER_PERSON - 1e-9)))


def bands_around(demand):
    # For a demand in a gap: (the most a team can take below it, (minimum, maximum) of the next band above it).
    teams = range(MIN_PEOPLE, PEOPLE_PER_CLUSTER + 1)
    below = [team_band(t)[1] for t in teams if team_band(t)[1] < demand]
    if MIN_PEOPLE == 1 and SOLO_MAX_DEMAND < demand:
        below.append(SOLO_MAX_DEMAND)
    above = [team_band(t) for t in teams if team_band(t)[0] > demand]
    return (max(below) if below else None), (min(above) if above else None)


def loads_of(labels):
    # {cluster label: demand}
    parts, inv, load, clat, clon = cluster_stats(labels)
    return dict(zip(parts.tolist(), load))


def km_to_cluster_centre(i, p, labels, flagged=None):
    # straight-line km from cabinet i to the centre of cluster p (flagged cabinets don't move a centre)
    return haversine_km(LAT[i], LON[i], *centre_of(members_of(labels, p), flagged))


def shed(labels, fallback, s, target, radius_km, keep_teams=False, fixed=None, exempt=None):
    # Pass cabinets of cluster s, one at a time, to touching clusters until s is at or below `target` tickets/day.
    # A cabinet only goes to a cluster whose new demand still fits a team (not the special case). Preferred:
    # clusters whose team stays the same, then a cabinet that gets s there at once, then the nearest centre.
    # keep_teams: only to clusters whose team stays the same. fixed: cabinets that must not move. exempt:
    # cabinets with no distance limit that don't move a centre (leftovers).
    # All or nothing: returns the new labels, or None if s can't get there.
    labels = labels.copy()
    flagged = fallback if exempt is None else fallback | exempt
    while True:
        load = loads_of(labels)
        if load[s] <= target + 1e-9:
            return labels
        ms = members_of(labels, s)
        opts = sorted((team_for(load[t] + DEM[i])[0] != team_for(load[t])[0], load[s] - DEM[i] > target + 1e-9,
                       km_to_cluster_centre(i, t, labels, flagged), i, t)
                      for i in ms if load[s] - DEM[i] >= SMALLEST_MIN - 1e-9 and (fixed is None or not fixed[i])
                      for t in touching_clusters(i, labels) if fits_band(load[t] + DEM[i])
                      and (not keep_teams or team_for(load[t] + DEM[i])[0] == team_for(load[t])[0]))
        choice = first_passing([(i, t) for *_, i, t in opts],
                               lambda it: {s: ms[ms != it[0]], it[1]: np.append(members_of(labels, it[1]), it[0])},
                               labels, fallback, radius_km, exempt)
        if choice is None:
            return None
        labels[choice[0]] = choice[1]


def top_up_to(labels, fallback, s, band, radius_km):
    # Take over cabinets from touching clusters until s reaches band = (minimum, maximum). A neighbour only gives a
    # cabinet away if its own team stays the same. All or nothing: returns the new labels, or None.
    labels = labels.copy()
    lo, hi = band
    while True:
        load = loads_of(labels)
        if load[s] >= lo - 1e-9:
            return labels
        ms = members_of(labels, s)
        opts = sorted((km_to_cluster_centre(i, s, labels, fallback), i, t)
                      for t in touching_clusters(ms, labels) for i in members_of(labels, t)
                      if load[s] + DEM[i] <= hi + 1e-9 and s in touching_clusters(i, labels)
                      and team_for(load[t] - DEM[i])[0] == team_for(load[t])[0])
        choice = first_passing([(i, t) for _, i, t in opts],
                               lambda it: {s: np.append(ms, it[0]),
                                           it[1]: members_of(labels, it[1])[members_of(labels, it[1]) != it[0]]},
                               labels, fallback, radius_km)
        if choice is None:
            return None
        labels[choice[0]] = s


def smaller_teams_step(labels, fallback, still_low, radius_km):
    # Give every cluster a team size. Clusters that reached MIN_DEMAND keep the full team; the others (the pool)
    # are tried with one person fewer each round. Clusters left between two bands then pass cabinets on or take
    # some over. Returns (labels, straight-line fallback flag per cabinet, team size per cluster label, labels of
    # the clusters to break up, log).
    labels = labels.copy()
    pool = {int(p) for p in still_low}
    log = []

    def load(p):
        return DEM[labels == p].sum()

    def note(p, result):
        m = members_of(labels, p)
        log.append({"n_cabinets": len(m), "demand": DEM[m].sum(),
                    "cabinets": ", ".join(cabinets["CabinetId"].iloc[m[:3]]) + (" ..." if len(m) > 3 else ""),
                    "result": result})

    for team in range(PEOPLE_PER_CLUSTER - 1, MIN_PEOPLE - 1, -1):
        if not pool:
            break
        lo, hi = team_band(team)
        labels, fallback, _, _ = min_demand_step(labels, hi, lo, radius_km, active=pool, fallback=fallback,
                                                 fallback_below=SMALLEST_MIN)
        pool &= {int(p) for p in np.unique(labels)}            # clusters merged away are gone
        for p in sorted(pool):
            if lo - 1e-9 <= load(p) <= hi + 1e-9:
                pool.discard(p)
                note(p, f"team of {team}")

    # what is left in the pool, lightest first: a gap (e.g. 7 to 8/day), the special case (6 to 7) or too small
    for p in sorted(pool, key=load):
        if not (labels == p).any():
            continue
        before = load(p)
        if team_for(before)[0] is None and before >= SMALLEST_MIN - 1e-9:   # in a gap: pass on, or take over
            below, above = bands_around(before)
            new = shed(labels, fallback, p, below, radius_km) if below is not None else None
            if new is None and above is not None:
                new = top_up_to(labels, fallback, p, above, radius_km)
            if new is not None:
                labels = new
        if team_for(load(p))[1]:                    # special case: try to get down to a normal one-person day
            new = shed(labels, fallback, p, team_band(1)[1], radius_km)
            if new is not None:
                labels = new
        team, special = team_for(load(p))
        changed = f" (was {before:.2f}/day)" if abs(load(p) - before) > 1e-9 else ""
        if team is None:
            note(p, "no team fits: broken up as leftovers" + changed)
        else:
            note(p, f"team of {team}" + (", special case" if special else "") + changed)

    # every cluster's team follows from its demand; clusters no team fits are broken up
    people, break_up = {}, []
    for p in np.unique(labels):
        team = team_for(load(p))[0]
        if team is None:
            break_up.append(int(p))
        else:
            people[int(p)] = team
    return labels, fallback, people, break_up, pd.DataFrame(log)


labels, fallback, people, break_up, team_log = smaller_teams_step(labels, fallback, still_low, RADIUS_KM)
save_cache_if_due(force=True)   # write the new road distances to the cache file
sizes = pd.Series(people).value_counts()
print("Team sizes so far: " + ", ".join(f"{sizes.get(t, 0)} cluster(s) of {t}"
                                        for t in range(PEOPLE_PER_CLUSTER, MIN_PEOPLE - 1, -1))
      + f" ({time.time() - START:.0f} s so far)")
if break_up:
    print(f"{len(break_up)} cluster(s) fit no team (too small, or between two bands): "
          "their cabinets are handled as leftovers below")
team_log.round(2)

## Leftovers – nearest cluster border
A cluster no team fits is broken up. Each of its cabinets joins the cluster with the nearest border (straight line to
the outline, touching clusters first) whose new demand still fits a team; that cluster's team changes if it has to.
Leftovers have no distance limit and are flagged `leftover`.

We prefer a cluster whose outline stays clear and whose own cabinets still keep their distance rule. If none does,
the nearest one is used and any of its cabinets pushed over their limit are flagged too, so nothing breaks a rule
without showing it. If no cluster has room at all, the cabinets stay together as their own cluster, marked
`below minimum`.

Centres moved while clusters changed, so at the end a fallback cabinet that is now within `RADIUS_KM` by road becomes
a normal cabinet again.

In [ ]:
def leftover_step(labels, fallback, people, break_up, cap, radius_km):
    # Break up the clusters in `break_up`, lightest first: each of their cabinets joins the cluster with the nearest
    # border (straight line) whose new demand still fits a team, and that cluster's team changes if it has to.
    # Returns (labels, team size per cluster label, leftover flag per cabinet, log).
    labels, people = labels.copy(), dict(people)
    leftover = np.zeros(len(DEM), dtype=bool)
    todo = {int(p) for p in break_up}
    borders, log = {}, []

    def km_to_centre(i, p):
        # straight line; leftovers and fallback cabinets don't move a centre
        return haversine_km(LAT[i], LON[i], *centre_of(members_of(labels, p), leftover | fallback))

    def km_to_border(i, p):
        # straight-line km from cabinet i to the outline of cluster p. Leftovers are not part of an outline, so it
        # stays the same while leftovers join.
        p = int(p)
        if p not in borders:
            m = members_of(labels, p)
            borders[p] = outline(m[~leftover[m]] if (~leftover[m]).any() else m)
        return borders[p].distance(shapely.Point(XY[i]))

    def clean(i, t, s):
        # t can take leftover i with the outlines clear and its own cabinets still passing their distance rule
        after = np.append(members_of(labels, t), i)
        exempt = leftover | (labels == s)    # leftovers have no distance limit and don't move the centre
        hidden = leftover | (labels == s)    # and are left out of the outlines
        return (outlines_clear({t: after, s: []}, labels, hidden)
                and groups_pass([after], fallback, radius_km, exempt)[0])

    def entry(i, **more):
        return {"CabinetId": cabinets["CabinetId"].iloc[i], "demand": DEM[i], **more}

    def special(d):
        return team_for(d)[1]

    while True:
        load = loads_of(labels)
        low = [p for p in todo if p in load and team_for(load[p])[0] is None]
        if not low:
            break
        s = min(low, key=load.get)
        while (labels == s).any():
            load = loads_of(labels)
            ms = members_of(labels, s)
            near = [(i, t) for i in ms for t in touching_clusters(i, labels)]
            anywhere = [(i, t) for i in ms for t in load if t != s]
            # only where the new demand still fits a team: touching clusters first, then anywhere; the one-person
            # special case (up to SOLO_MAX_DEMAND) only if no normal band is possible
            opts = []
            for pairs, fits in ((near, fits_band), (near, special), (anywhere, fits_band), (anywhere, special)):
                opts = [(i, t) for i, t in pairs if fits(load[t] + DEM[i]) and load[t] + DEM[i] <= cap + 1e-9]
                if opts:
                    break
            if not opts:   # no cluster has room: the rest stay together as their own small cluster
                todo.discard(s)
                for i in ms:
                    leftover[i] = True
                    log.append(entry(i, km_to_border=0.0, straight_km_to_new_centre=km_to_centre(i, s),
                                     rules_kept=False, others_pushed_over_limit="",
                                     note="no cluster has room: kept as own cluster"))
                break
            opts.sort(key=lambda it: km_to_border(*it))
            choice = next(((i, t) for i, t in opts if clean(i, t, s)), None)
            i, t = choice if choice is not None else opts[0]
            t, km = int(t), km_to_border(i, t)
            labels[i], leftover[i] = t, True
            pushed = []
            if choice is None:   # the nearest cluster had to be used: flag its cabinets pushed over their limit
                m = members_of(labels, t)
                centre = centre_of(m, leftover | fallback)
                m = m[~leftover[m]]
                straight = haversine_km(LAT[m], LON[m], *centre)
                road = road_km([(centre, (LAT[j], LON[j])) for j in m])
                over = m[np.where(fallback[m], straight > radius_km, road > radius_km)]
                leftover[over] = True
                borders.pop(t, None)
                pushed = list(cabinets["CabinetId"].iloc[over])
            team, is_special = team_for(load[t] + DEM[i])
            notes = []
            if people.get(t) is not None and team != people[t]:
                notes.append(f"team changed from {people[t]} to {team}")
            if is_special:
                notes.append("one person, special case")
            people[t] = team
            log.append(entry(i, km_to_border=km, straight_km_to_new_centre=km_to_centre(i, t),
                             rules_kept=choice is not None, others_pushed_over_limit=", ".join(pushed),
                             note="; ".join(notes)))
    # every cluster's final team follows from its demand
    return labels, {p: smallest_team(d) for p, d in loads_of(labels).items()}, leftover, pd.DataFrame(log)


def settle_fallback(labels, fallback, leftover, radius_km):
    # Centres moved since the fallback cabinets were flagged. A fallback cabinet that is now within radius_km by
    # road of its centre becomes a normal cabinet again, nearest first, as long as counting it in the centre keeps
    # every cabinet of its cluster within its limit. Returns the new fallback flags.
    fallback = fallback & ~leftover
    for p in np.unique(labels[fallback]):
        while True:
            m = members_of(labels, p)
            cand = m[fallback[m]]
            if len(cand) == 0:
                break
            centre = centre_of(m, fallback | leftover)
            km = road_km([(centre, (LAT[i], LON[i])) for i in cand])
            back = None
            for i in cand[np.argsort(km)][np.sort(km) <= radius_km + 1e-9]:
                trial = fallback.copy()
                trial[i] = False
                if groups_pass([m], trial, radius_km, leftover)[0]:
                    back = trial
                    break
            if back is None:
                break
            fallback = back
    return fallback


labels, people, leftover, leftover_log = leftover_step(labels, fallback, people, break_up, CAP, RADIUS_KM)
n_flagged = int((fallback & ~leftover).sum())
fallback = settle_fallback(labels, fallback, leftover, RADIUS_KM)
print(f"{n_flagged - int(fallback.sum())} fallback cabinet(s) are within {RADIUS_KM:g} km by road of their final "
      "centre after all: normal cabinets again")
save_cache_if_due(force=True)   # write the new road distances to the cache file
print(f"{int(leftover.sum())} cabinet(s) flagged as leftover, {int(fallback.sum())} as straight-line fallback; "
      f"{len(np.unique(labels))} clusters ({time.time() - START:.0f} s so far)")
leftover_log.round(2)

## Save people – one person fewer where a little demand can move
Teams are rounded up: a cluster just over a multiple of `DEMAND_PER_PERSON` costs a whole extra person. We take those
clusters, smallest excess first, and pass cabinets on to touching clusters until one person fewer can carry it.

* A neighbour only takes a cabinet if its own team stays the same, so every success saves a person.
* All or nothing: if the cluster can't get all the way down, nothing changes.
* The road rule, the outlines and `CAP` still hold, and flagged cabinets stay where they are.

In [ ]:
def people_needed(labels):
    # Total people per day: every cluster gets the team its demand calls for.
    return sum(smallest_team(d) for d in loads_of(labels).values())


def trim_people(labels, fallback, leftover, radius_km):
    # Clusters a little over a multiple of DEMAND_PER_PERSON pass cabinets on to touching clusters whose team stays
    # the same, until one person fewer can carry them; smallest excess first. Every success saves at least one
    # person, so the loop ends. Fallback and leftover cabinets don't move. Returns (labels, log).
    labels, log, tried = labels.copy(), [], set()
    fixed = fallback | leftover
    while True:
        load = loads_of(labels)
        options = sorted((d - team_band(team - 1)[1], s, team) for s, d in load.items()
                         for team in [team_for(d)[0]] if team is not None and team > MIN_PEOPLE and s not in tried)
        for excess, s, team in options:
            new = shed(labels, fallback, s, team_band(team - 1)[1], radius_km, keep_teams=True, fixed=fixed,
                       exempt=leftover)
            if new is not None and people_needed(new) < people_needed(labels):
                m = members_of(labels, s)
                log.append({"n_cabinets": len(m), "cabinets": ", ".join(cabinets["CabinetId"].iloc[m[:3]])
                            + (" ..." if len(m) > 3 else ""), "demand_before": load[s],
                            "demand_after": loads_of(new)[s], "team_before": team,
                            "team_after": smallest_team(loads_of(new)[s]),
                            "cabinets_passed_on": int(((labels == s) & (new != s)).sum())})
                labels = new
                tried.clear()   # something changed: every cluster gets another try
                break
            tried.add(s)
        else:
            return labels, pd.DataFrame(log)


people_before_trim = people_needed(labels)
labels, trim_log = trim_people(labels, fallback, leftover, RADIUS_KM)
fallback = settle_fallback(labels, fallback, leftover, RADIUS_KM)
save_cache_if_due(force=True)   # write the new road distances to the cache file
TRIM_SAVED = people_before_trim - people_needed(labels)
print(f"{len(trim_log)} cluster(s) passed cabinets on and need one person fewer: {people_before_trim} -> "
      f"{people_needed(labels)} people per day ({time.time() - START:.0f} s so far)")
trim_log.round(2)


## Fit the people available
The first run rarely needs exactly `TOTAL_PEOPLE`, so the demand is scaled again and the whole model (Demand Layer to
Save people) runs again:

1. while no run fits, scale **down** in proportion: `scale × TOTAL_PEOPLE / people needed`;
2. while every run fits but leaves people free, scale **up** the same way, up to `MAX_DEMAND_SCALE`;
3. once one run fits and a bigger one doesn't, try halfway between them.

After at most `FIT_ROUNDS` extra runs we keep the run that fits with the most demand. If none fits, the run with the
fewest people is kept with a warning. Then every cabinet's road distance to its final centre is worked out for the
tables and the map.

In [ ]:
def set_demand(scale):
    # Use this share of the raw demand everywhere. DEM is shared by every step, so it is changed in place.
    global DEMAND_SCALE, TOTAL_DEMAND, K_MIN, K_MAX
    DEM[:] = cabinets["raw_demand"].to_numpy() * scale
    cabinets["demand"] = DEM.copy()
    DEMAND_SCALE, TOTAL_DEMAND = scale, DEM.sum()
    K_MIN = math.ceil(TOTAL_DEMAND / CAP - 1e-9)
    K_MAX = max(K_MIN, math.floor(TOTAL_DEMAND / SMALLEST_MIN + 1e-9))


def run_model():
    # Every step from the Demand Layer to Save people, without the printing.
    # Returns (labels, straight-line fallback flags, leftover flags, people saved by trimming).
    labels = demand_layer(CAP, RADIUS_KM, SEED, BALANCE_TOL)[0]
    labels = distance_layer(labels, CAP, RADIUS_KM, SEED)[0]
    labels, fallback, _, still_low = min_demand_step(labels, CAP, MIN_DEMAND, RADIUS_KM, fallback_below=SMALLEST_MIN)
    labels, fallback, people, break_up, _ = smaller_teams_step(labels, fallback, still_low, RADIUS_KM)
    labels, people, leftover, _ = leftover_step(labels, fallback, people, break_up, CAP, RADIUS_KM)
    fallback = settle_fallback(labels, fallback, leftover, RADIUS_KM)
    before = people_needed(labels)
    labels = trim_people(labels, fallback, leftover, RADIUS_KM)[0]
    fallback = settle_fallback(labels, fallback, leftover, RADIUS_KM)
    save_cache_if_due(force=True)   # write the new road distances to the cache file
    return labels, fallback, leftover, before - people_needed(labels)


# every run: {scale: (people needed, labels, fallback, leftover, people saved by trimming)}
runs = {DEMAND_SCALE: (people_needed(labels), labels, fallback, leftover, TRIM_SAVED)}


def report(n, scale):
    need, labels_, *_ = runs[scale]
    print(f"run {n}: demand {scale:.2%} of the data ({DEM.sum():.2f} tickets/day), "
          f"{len(np.unique(labels_))} clusters, {need} people" + ("" if TOTAL_PEOPLE is None else
                                                               " - fits" if need <= TOTAL_PEOPLE else " - too many"))


report(0, DEMAND_SCALE)
if TOTAL_PEOPLE is not None:
    for n in range(1, FIT_ROUNDS + 1):
        fits = [s for s, r in runs.items() if r[0] <= TOTAL_PEOPLE]
        over = [s for s, r in runs.items() if r[0] > TOTAL_PEOPLE]
        if not fits:
            low = min(over)
            scale = low * TOTAL_PEOPLE / runs[low][0]       # scale down in proportion to the people needed
        else:
            best = max(fits)
            above = [s for s in over if s > best]
            if not above:
                # every run so far fits: try more demand, in proportion to the people still free
                scale = min(MAX_DEMAND_SCALE, best * TOTAL_PEOPLE / runs[best][0])
                if scale - best < 0.005:   # nothing more to gain (within half a percent), or at MAX_DEMAND_SCALE
                    break
            elif min(above) - best < 0.005:  # nothing better to find (within half a percent)
                break
            else:
                scale = (best + min(above)) / 2
        set_demand(scale)
        new_labels, new_fallback, new_leftover, saved = run_model()
        runs[scale] = (people_needed(new_labels), new_labels, new_fallback, new_leftover, saved)
        report(n, scale)
    fits = [s for s, r in runs.items() if r[0] <= TOTAL_PEOPLE]
    chosen = max(fits) if fits else min(runs, key=lambda s: (runs[s][0], -s))
    if not fits:
        print(f"WARNING: no run fits {TOTAL_PEOPLE} people within FIT_ROUNDS = {FIT_ROUNDS}: keeping the run with "
              "the fewest people. Raise FIT_ROUNDS and run again.")
else:
    chosen = DEMAND_SCALE
set_demand(chosen)
need, labels, fallback, leftover, TRIM_SAVED = runs[chosen]
FIT_RUN_COUNT = len(runs) - 1
people = {p: smallest_team(d) for p, d in loads_of(labels).items()}

# final road distances for every cabinet (no shortcut), so leftovers get their real road km too. The centre leaves
# out leftovers and fallback cabinets.
road, straight = road_to_centre(labels, np.inf, leftover | fallback)
save_cache_if_due(force=True)   # write the new road distances to the cache file
rule = np.where(leftover, "leftover", np.where(fallback, "straight-line fallback", "road"))
print(f"Kept: demand {DEMAND_SCALE:.2%} of the data ({TOTAL_DEMAND:.2f} scaled, {RAW_TOTAL_DEMAND:.2f} real tickets/day), "
      f"{len(np.unique(labels))} clusters, {need} people per day"
      + ("" if TOTAL_PEOPLE is None else f" (available: {TOTAL_PEOPLE})")
      + f"; {int(leftover.sum())} leftover, {int(fallback.sum())} straight-line fallback cabinet(s) "
      f"({time.time() - START:.0f} s so far)")
if TOTAL_PEOPLE is not None and need < TOTAL_PEOPLE and DEMAND_SCALE >= MAX_DEMAND_SCALE - 1e-9:
    print(f"{TOTAL_PEOPLE - need} of {TOTAL_PEOPLE} people are not needed even at MAX_DEMAND_SCALE = "
          f"{MAX_DEMAND_SCALE:g}: they are spare")

## Cluster tables
One row per cluster (`C001`, `C002`, ...) and one per cabinet. `people` is the team size. Demand columns are scaled;
the `real_` columns are the same in real tickets (divided by `DEMAND_SCALE`). `demand_rule` is `ok`,
`one person, special case`, or `below minimum` for a cluster of leftovers no one could take. The centre leaves out
flagged cabinets. We also report how far the centres had to be snapped to a road, and how much longer road distances
are than straight lines.

In [ ]:
def build_tables(labels, road, straight, rule, people):
    # Turn cluster labels into the clusters / assignments tables. rule is each cabinet's distance rule:
    # "road", "straight-line fallback" or "leftover".
    cluster_rows, assign_rows = [], []
    for n, p in enumerate(np.unique(labels), start=1):
        cid = f"C{n:03d}"
        m = labels == p
        demand, team = DEM[m].sum(), people[int(p)]
        fits, special = team_for(demand)
        lo, hi = team_band(team)
        hi = SOLO_MAX_DEMAND if special else hi
        clat, clon = centre_of(np.where(m)[0], rule != "road")   # flagged cabinets don't move the centre
        cluster_rows.append({"cluster_id": cid, "n_cabinets": int(m.sum()), "people": team,
                             "demand_per_day": demand, "min_demand": lo, "max_demand": hi,
                             "demand_rule": ("below minimum" if fits is None
                                             else "one person, special case" if special else "ok"),
                             "fill_rate": demand / hi, "demand_per_person": demand / team,
                             "max_road_dist_to_centre_km": road[m].max(),
                             "max_straight_dist_to_centre_km": straight[m].max(),
                             "centre_lat": clat, "centre_long": clon,
                             "n_straight_line_fallback": int((rule[m] == "straight-line fallback").sum()),
                             "n_leftover": int((rule[m] == "leftover").sum()),
                             "real_demand_per_day": demand / DEMAND_SCALE,
                             "real_demand_per_person": demand / team / DEMAND_SCALE})
        for i in np.where(m)[0]:
            assign_rows.append({"CabinetId": cabinets["CabinetId"].iloc[i], "cluster_id": cid, "people": team,
                                "allocated_demand": DEM[i], "road_dist_to_centre_km": road[i],
                                "straight_dist_to_centre_km": straight[i], "distance_rule": rule[i],
                                "real_demand": DEM[i] / DEMAND_SCALE})
    return pd.DataFrame(cluster_rows), pd.DataFrame(assign_rows)


clusters, assignments = build_tables(labels, road, straight, rule, people)

snaps = pd.Series([snap_km(la, lo) for la, lo in zip(clusters["centre_lat"], clusters["centre_long"])],
                  index=clusters["cluster_id"])
print(f"Centre snapping to the nearest road: median {snaps.median() * 1000:.0f} m, max {snaps.max() * 1000:.0f} m "
      f"({snaps.idxmax()})")
far = snaps[snaps > SNAP_WARN_KM]
if len(far):
    print(f"WARNING: {len(far)} cluster centre(s) are more than {SNAP_WARN_KM} km from the nearest road, so their "
          "road distances start from a point some way off: " + ", ".join(far.index))

away = (straight >= 1) & np.isfinite(road)   # very short distances give misleading ratios
ratio = road[away] / straight[away]
print(f"Road / straight-line distance (cabinets at least 1 km from their centre): mean {ratio.mean():.2f}, "
      f"median {np.median(ratio):.2f}, range {ratio.min():.2f} to {ratio.max():.2f}")
clusters.head(10).round(3)

## Validation
Hard checks, recomputed from the tables. The notebook stops if any fails:

1. All demand is allocated, for every cabinet and in total.
2. Every cluster has the team its demand calls for; only a cluster of leftovers may fit none.
3. The centre is the average of the road-rule cabinets. Those are within `RADIUS_KM` by road, fallback cabinets within
   `RADIUS_KM` in a straight line, leftovers anywhere.
4. Every cabinet is in exactly one cluster, and no cluster is empty.
5. No two outlines overlap (leftovers left out).

In [ ]:
def validate(cabinets, clusters, assignments, radius_km):
    # 1. demand conservation
    assert abs(assignments["allocated_demand"].sum() - cabinets["demand"].sum()) < TOL, \
        "allocated demand != input demand"
    per_cab = assignments.groupby("CabinetId")["allocated_demand"].sum()
    gap = per_cab.reindex(cabinets["CabinetId"]).fillna(0).values - cabinets["demand"].values
    assert np.abs(gap).max() < TOL, "some cabinet's demand is not fully allocated"
    # 2. team size and demand (the clusters table must agree): every team is the one its demand calls for
    load = assignments.groupby("cluster_id")["allocated_demand"].sum()
    info = clusters.set_index("cluster_id").reindex(load.index)
    team = info["people"]
    assert team.between(MIN_PEOPLE, PEOPLE_PER_CLUSTER).all(), "a cluster's team size is out of range"
    fit = [team_for(d) for d in load]
    assert (team == [smallest_team(d) for d in load]).all(), "a cluster's team is not the one its demand calls for"
    expected = ["below minimum" if t is None else "one person, special case" if sp else "ok" for t, sp in fit]
    assert (info["demand_rule"] == expected).all(), "demand_rule does not match the cluster's demand"
    low = info["demand_rule"] == "below minimum"
    special = info["demand_rule"] == "one person, special case"
    hi = np.where(special, SOLO_MAX_DEMAND, team * DEMAND_PER_PERSON)
    assert (load[~low] >= team[~low] * MIN_DEMAND_PER_PERSON - TOL).all(), "a cluster is below its team's minimum"
    assert (load[~low] <= hi[~low] + TOL).all(), "a cluster is above its team's maximum"
    assert (team[special] == 1).all(), "the special case is for one person only"
    # only a cluster made of leftovers that no cluster had room for may fit no team
    all_left = assignments.groupby("cluster_id")["distance_rule"].apply(lambda r: (r == "leftover").all())
    assert all_left.reindex(load.index)[low].all(), "a cluster fits no team but is not made of leftovers"
    assert (assignments["people"] == assignments["cluster_id"].map(team)).all(), "assignments table is stale"
    assert np.allclose(load.reindex(clusters["cluster_id"]), clusters["demand_per_day"]), "clusters table is stale"
    # 3. distance to the (recomputed) centre: road for "road", straight line for "straight-line fallback".
    #    The centre comes from the "road" cabinets only (a cluster without any uses all its cabinets).
    m = assignments.merge(cabinets[["CabinetId", "cabinetlat", "cabinetlong"]], on="CabinetId")
    assert m["distance_rule"].isin(["road", "straight-line fallback", "leftover"]).all(), "unknown distance_rule"
    by_road = (m["distance_rule"] == "road").to_numpy()
    by_line = (m["distance_rule"] == "straight-line fallback").to_numpy()
    every = m.groupby("cluster_id")[["cabinetlat", "cabinetlong"]].mean()
    centres = m[by_road].groupby("cluster_id")[["cabinetlat", "cabinetlong"]].mean().reindex(every.index).fillna(every)
    table = clusters.set_index("cluster_id")[["centre_lat", "centre_long"]].reindex(centres.index)
    assert np.allclose(table.to_numpy(), centres.to_numpy()), "cluster centres must leave out flagged cabinets"
    centre = centres.reindex(m["cluster_id"])
    road = road_km(list(zip(zip(centre["cabinetlat"], centre["cabinetlong"]),
                            zip(m["cabinetlat"], m["cabinetlong"]))))
    straight = haversine_km(m["cabinetlat"].to_numpy(), m["cabinetlong"].to_numpy(),
                            centre["cabinetlat"].to_numpy(), centre["cabinetlong"].to_numpy())
    too_far = m.loc[by_road & (road > radius_km + 1e-9), "CabinetId"].tolist()
    assert not too_far, f"cabinet(s) more than RADIUS_KM by road from their cluster centre: {too_far}"
    too_far = m.loc[by_line & (straight > radius_km + 1e-9), "CabinetId"].tolist()
    assert not too_far, f"fallback cabinet(s) more than RADIUS_KM in a straight line: {too_far}"
    # 4. structure
    assert (clusters["n_cabinets"] > 0).all() and (clusters["demand_per_day"] > 0).all(), "empty cluster"
    assert set(clusters["cluster_id"]) == set(assignments["cluster_id"]), "cluster/assignment mismatch"
    counts = assignments["CabinetId"].value_counts()
    assert counts.reindex(cabinets["CabinetId"]).eq(1).all() and len(counts) == len(cabinets), \
        "a cabinet is not in exactly one cluster"
    # 5. no overlapping outlines, recomputed from scratch for every pair (leftovers left out)
    in_outline = m["distance_rule"] != "leftover"
    xy = project_km(m["cabinetlat"], m["cabinetlong"])
    ids, shapes = [], []
    for cid, rows in m[in_outline].groupby("cluster_id").groups.items():
        ids.append(cid)
        shapes.append(shapely.MultiPoint(xy[m.index.get_indexer(rows)]).convex_hull)
    a, b = shapely.STRtree(shapes).query(shapes, predicate="intersects")
    clash = sorted({tuple(sorted((ids[x], ids[y]))) for x, y in zip(a, b)
                    if x < y and not shapes[x].touches(shapes[y])})
    assert not clash, f"overlapping cluster outlines: {clash[:10]}"
    print(f"all checks passed: {len(clusters)} clusters ({int(low.sum())} below minimum, "
          f"{int(special.sum())} one-person special case), "
          f"{len(assignments)} cabinets ({int(by_line.sum())} "
          f"straight-line fallback, {int((~by_road & ~by_line).sum())} leftover), demand per cluster "
          f"{load.min():.2f} to {load.max():.2f}, max road distance to centre {road[by_road].max():.2f} km, "
          "no overlapping outlines")


validate(cabinets, clusters, assignments, RADIUS_KM)

## Secondary cabinets
A cluster's own cabinets are its **primary** cabinets. Cabinets of other clusters within `RADIUS_KM` by road of its
centre are its **secondary** cabinets: backup work nearby, for example when a team finishes early. They are a list
only and don't count in the demand. A cabinet is primary in one cluster but can be secondary for several. The
nearest `MAX_SECONDARY` per cluster are kept. They go to `secondaries.csv` and to the columns `n_secondary` and
`secondary_demand_per_day` of `clusters.csv`.

In [ ]:
def find_secondaries(clusters, assignments, radius_km, max_n):
    # The nearest max_n (cluster, cabinet) pairs per cluster where the cabinet belongs to another cluster but lies within radius_km BY ROAD
    # (centre -> cabinet) of this cluster's centre. Straight-line shortcut first: only pairs within radius_km in a
    # straight line are sent to OSRM.
    primary = assignments.set_index("CabinetId")["cluster_id"].reindex(cabinets["CabinetId"]).to_numpy()
    cids = clusters["cluster_id"].to_numpy()
    clat, clon = clusters["centre_lat"].to_numpy(), clusters["centre_long"].to_numpy()
    straight = haversine_km(clat[:, None], clon[:, None], LAT[None, :], LON[None, :])   # clusters x cabinets
    ci, i = np.nonzero(straight <= radius_km + 1e-9)
    keep = cids[ci] != primary[i]
    ci, i = ci[keep], i[keep]
    road = road_km([((clat[c], clon[c]), (LAT[j], LON[j])) for c, j in zip(ci, i)])
    ok = road <= radius_km + 1e-9
    ci, i, road = ci[ok], i[ok], road[ok]
    found = (pd.DataFrame({"cluster_id": cids[ci], "CabinetId": cabinets["CabinetId"].to_numpy()[i],
                          "primary_cluster_id": primary[i], "demand": DEM[i], "road_dist_to_centre_km": road,
                          "straight_dist_to_centre_km": straight[ci, i]})
            .sort_values(["cluster_id", "road_dist_to_centre_km"]))
    if max_n is not None:
        found = found.groupby("cluster_id").head(max_n)   # keep only the nearest max_n per cluster
    return found.reset_index(drop=True)

#Made by Iman Thaqif - Intern @ NAS, August 26 - Oct 26.
requests_before = OSRM_STATS["requests"]

secondaries = find_secondaries(clusters, assignments, RADIUS_KM, MAX_SECONDARY)
per_cluster = secondaries.groupby("cluster_id")
clusters["n_secondary"] = clusters["cluster_id"].map(per_cluster.size()).fillna(0).astype(int)
clusters["secondary_demand_per_day"] = clusters["cluster_id"].map(per_cluster["demand"].sum()).fillna(0.0)

# checks: a cabinet is never secondary for its own cluster, and every secondary is within the radius (recomputed)
assert (secondaries["cluster_id"] != secondaries["primary_cluster_id"]).all(), "a cabinet is secondary for its own cluster"
check = secondaries.merge(clusters[["cluster_id", "centre_lat", "centre_long"]], on="cluster_id") \
                   .merge(cabinets[["CabinetId", "cabinetlat", "cabinetlong"]], on="CabinetId")
road = road_km(list(zip(zip(check["centre_lat"], check["centre_long"]), zip(check["cabinetlat"], check["cabinetlong"]))))
assert (road <= RADIUS_KM + 1e-9).all(), "a secondary cabinet is more than RADIUS_KM by road"
save_cache_if_due(force=True)   # write the new road distances to the cache file
print(f"{len(secondaries)} secondary links: {int((clusters['n_secondary'] > 0).sum())} of {len(clusters)} clusters have "
      f"secondary cabinets, on average {clusters['n_secondary'].mean():.1f} per cluster "
      f"({secondaries['CabinetId'].nunique()} different cabinets); "
      f"{OSRM_STATS['requests'] - requests_before} OSRM requests for this step")
clusters[["cluster_id", "n_cabinets", "demand_per_day", "n_secondary", "secondary_demand_per_day"]].head(10).round(2)

### Result summary
* **Clusters** against the possible range (see *Load and check the data*): closer to the lower end means fewer days of
  work.
* **People needed per day**: all team sizes added up; the list above the table gives each cluster's team, also saved
  as `teams.csv`.
* **Fill rate**: cluster demand / its team's maximum. **Demand per person**: cluster demand / team size.
* **Real tickets per person**: the same in real tickets, after scaling.
* **Flagged cabinets**: straight-line fallbacks and leftovers.

In [ ]:
fr = clusters["fill_rate"]
loads = clusters["demand_per_day"]
real = clusters["real_demand_per_person"]   # real tickets per person (scaled demand / DEMAND_SCALE)
by_road = assignments["distance_rule"] == "road"
teams = clusters["people"].value_counts()
team_list = [f"{c} : {n} {'Person' if n == 1 else 'People'}" for c, n in zip(clusters["cluster_id"], clusters["people"])]
print(f"People needed per day: {int(clusters['people'].sum())}"
      + ("" if TOTAL_PEOPLE is None else f" (available: {TOTAL_PEOPLE})"))
if TOTAL_PEOPLE is not None and clusters["people"].sum() > TOTAL_PEOPLE:
    print(f"WARNING: the clusters need {int(clusters['people'].sum()) - TOTAL_PEOPLE} more people than "
          f"TOTAL_PEOPLE - not every person can be given a full {DEMAND_PER_PERSON} tickets/day")
for k in range(0, len(team_list), 6):
    print("    ".join(f"{t:<15}" for t in team_list[k:k + 6]))
pd.Series({
    "Clusters": f"{len(clusters)}",
    "Possible range": f"{K_MIN} to {K_MAX}",
    "Clusters by team size": ", ".join(f"{teams.get(t, 0)} of {t}"
                                       for t in range(PEOPLE_PER_CLUSTER, MIN_PEOPLE - 1, -1)),
    "People needed per day": f"{int(clusters['people'].sum())} "
                             f"(all full teams: {len(clusters) * PEOPLE_PER_CLUSTER})",
    "People available (TOTAL_PEOPLE)": "-" if TOTAL_PEOPLE is None else f"{TOTAL_PEOPLE}",
    "People saved by Save people": f"{TRIM_SAVED}",
    "Extra runs to fit TOTAL_PEOPLE": f"{FIT_RUN_COUNT}",
    "Planning demand (tickets/day)": f"{TOTAL_DEMAND:.2f} = {DEMAND_SCALE:.1%} of the real {RAW_TOTAL_DEMAND:.2f}",
    "Real tickets per person per day": f"mean {real.mean():.2f}, {real.min():.2f} to {real.max():.2f}",
    "People not used (spare)": "-" if TOTAL_PEOPLE is None else f"{max(0, TOTAL_PEOPLE - int(clusters['people'].sum()))}",
    "Biggest team / tickets per person per day":
        f"{PEOPLE_PER_CLUSTER} / {MIN_DEMAND_PER_PERSON} to {DEMAND_PER_PERSON}",
    "Mean fill rate": f"{fr.mean():.1%}",
    "Min fill rate": f"{fr.min():.1%}",
    "Mean demand per person (tickets/day)": f"{clusters['demand_per_person'].mean():.2f}",
    "One-person special case (up to SOLO_MAX_DEMAND)":
        f"{int((clusters['demand_rule'] == 'one person, special case').sum())}",
    "Clusters below their minimum": f"{int((clusters['demand_rule'] == 'below minimum').sum())}",
    "Worst demand gap between clusters (tickets/day)": f"{loads.max() - loads.min():.2f}",
    "Max road distance to centre, road-rule cabinets (km)":
        f"{assignments.loc[by_road, 'road_dist_to_centre_km'].max():.2f}",
    "Max straight-line distance to centre (km)": f"{clusters['max_straight_dist_to_centre_km'].max():.2f}",
    "Straight-line fallback cabinets": f"{int((assignments['distance_rule'] == 'straight-line fallback').sum())}",
    "Leftover cabinets": f"{int((assignments['distance_rule'] == 'leftover').sum())}",
    "Road distances from": ("APPROXIMATE: OSRM unreachable, straight-line x 1.4" if OSRM_STATS["fallback"]
                            else f"OSRM ({OSRM_PROFILE}), {OSRM_STATS['requests']} requests this run"),
    "Secondary cabinet links": f"{len(secondaries)}",
    "Mean secondary cabinets per cluster": f"{clusters['n_secondary'].mean():.1f}",
    "Run time (s)": f"{time.time() - START:.0f}",
}, name="value").to_frame()

## Save outputs
Written to `OUTPUT_FOLDER`:

* `clusters.csv`: one row per cluster with its team, demand (scaled and real), fill rate, farthest cabinet, centre
  and flagged cabinets.
* `teams.csv`: the team per cluster (`C001 : 5 People`) and the total people per day.
* `assignments.csv`: one row per cabinet with its cluster, demand, distances to the centre and `distance_rule`
  (`road`, `straight-line fallback` or `leftover`).
* `secondaries.csv`: one row per cluster and secondary cabinet.
* `rules.csv`: the partition, the rules used and the `DEMAND_SCALE` kept.

In [ ]:
def team_table(clusters):
    # One row per cluster with its team ("C001 : 3 People"), and a last row with the total people per day.
    teams = clusters[["cluster_id", "people", "demand_per_day", "demand_rule"]].copy()
    teams.insert(1, "team", [f"{c} : {n} {'Person' if n == 1 else 'People'}"
                             for c, n in zip(teams["cluster_id"], teams["people"])])
    total = pd.DataFrame([{"cluster_id": "Total", "team": f"{int(teams['people'].sum())} People per day",
                           "people": int(teams["people"].sum()), "demand_per_day": teams["demand_per_day"].sum(),
                           "demand_rule": ""}])
    return pd.concat([teams, total], ignore_index=True)
#Made by Iman Thaqif - Intern @ NAS, August 26 - Oct 26.

def save_tables(clusters, assignments, secondaries):
    clusters.round(4).to_csv(OUT_DIR / "clusters.csv", index=False)
    team_table(clusters).round(4).to_csv(OUT_DIR / "teams.csv", index=False)
    assignments.round(4).to_csv(OUT_DIR / "assignments.csv", index=False)
    secondaries.round(4).to_csv(OUT_DIR / "secondaries.csv", index=False)
    pd.DataFrame([{"partition": RUN_PARTITION, **RULES_USED, "DEMAND_SCALE": DEMAND_SCALE}]).to_csv(
        OUT_DIR / "rules.csv", index=False)


save_tables(clusters, assignments, secondaries)
print(f"saved clusters.csv ({len(clusters)} rows), teams.csv, assignments.csv ({len(assignments)} rows), "
      f"secondaries.csv ({len(secondaries)} rows), rules.csv")

## Interactive map
`map.html` in `OUTPUT_FOLDER`, also shown below. Layers can be switched on and off at the top right.

* **Outlines** around each cluster's road-rule cabinets; dashed for a smaller team.
* **Territories**: the map shaded by the cluster of the nearest cabinet.
* **Centres** with the team size in them (dashed ring: one-person special case), and the cluster names next to them.
* **Cabinets** in their cluster's colour, bigger when busier. Flagged cabinets are grey: a navy ring for a
  fallback, a red ring for a leftover. A separate layer (off at first) shows them in their cluster's colour.
* **`RADIUS_KM` circles** (straight line) and **secondary links** (off at first).

Popups give the cabinet's or centre's position, demand and distances; with scaled demand both the scaled and the
real numbers are shown.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
FLAGGED_FILL = "#9aa0a6"   # flagged cabinets are grey, not in their cluster's colour
RINGS = {"road": ("#ffffff", 1), "straight-line fallback": ("#150080", 3), "leftover": ("#d00000", 3)}


def colour_clusters(areas):
    # Greedy colouring: clusters whose territories touch get different colours (most neighbours first).
    ids, shapes = list(areas), list(areas.values())
    a, b = shapely.STRtree(shapes).query(shapes, predicate="intersects")
    neighbours = {c: [] for c in ids}
    for x, y in zip(a, b):
        if x != y:
            neighbours[ids[x]].append(ids[y])
    colours = {}
    for c in sorted(ids, key=lambda c: -len(neighbours[c])):
        taken = {colours[d] for d in neighbours[c] if d in colours}
        free = [p for p in PALETTE if p not in taken]
        colours[c] = free[0] if free else PALETTE[len(colours) % len(PALETTE)]
    return colours


def demand_text(x, digits=2, unit="/day"):
    # Demand as the model used it, with the real value next to it when the demand was scaled to fit TOTAL_PEOPLE.
    if abs(DEMAND_SCALE - 1) < 1e-9:
        return f"{x:.{digits}f}{unit}"
    return f"{x:.{digits}f}{unit} (scaled) · {x / DEMAND_SCALE:.{digits}f}{unit} (real)"


def people_text(n):
    return f"{n} {'person' if n == 1 else 'people'}"


def centre_icon(team, colour, special):
    # A large round marker in the cluster's colour with the team size in it (dashed ring = one-person special case).
    ring = "dashed" if special else "solid"
    return folium.DivIcon(icon_size=(32, 32), icon_anchor=(16, 16),
                          html=f"<div style='width:24px;height:24px;border-radius:50%;background:{colour};"
                               f"border:4px {ring} #fff;box-shadow:0 0 0 2px #111,0 2px 8px rgba(0,0,0,.6);"
                               f"color:#fff;font:bold 15px/24px sans-serif;text-align:center;"
                               f"text-shadow:0 0 3px #000'>{team}</div>")


def name_icon(cid, team):
    # The cluster's name and team size, next to its centre.
    return folium.DivIcon(icon_size=(150, 20), icon_anchor=(-20, 10),
                          html=f"<div style='display:inline-block;white-space:nowrap;background:#fff;"
                               f"border:1px solid #111;border-radius:3px;padding:0 5px;"
                               f"font:bold 12px/18px sans-serif;color:#111'>{cid} : {people_text(team)}</div>")


def make_map(clusters, assignments, secondaries, radius_km, people, min_people):
    fmap = folium.Map(location=[REF_LAT, REF_LON], zoom_start=11, tiles="OpenStreetMap")
    by_cab = assignments.set_index("CabinetId").reindex(cabinets["CabinetId"])
    cluster_of = by_cab["cluster_id"].to_numpy()
    flagged = (by_cab["distance_rule"] != "road").to_numpy()   # straight-line fallback and leftover
    areas = cluster_areas(cluster_of, flagged)          # flagged cabinets are not part of a cluster's territory
    colours = colour_clusters(areas)
    for cid in clusters["cluster_id"]:
        colours.setdefault(cid, FLAGGED_FILL)           # a cluster made only of flagged cabinets
    territory_layer = folium.FeatureGroup(name="Territories")
    outline_layer = folium.FeatureGroup(name="Cluster outlines")
    circle_layer = folium.FeatureGroup(name=f"{radius_km:g} km straight-line circles", show=False)
    centre_layer = folium.FeatureGroup(name="Cluster centres")
    name_layer = folium.FeatureGroup(name="Cluster names")
    dot_layer = folium.FeatureGroup(name="Cabinets (coloured by cluster)")
    flag_layer = folium.FeatureGroup(name="Fallback and leftover cabinets (drawn apart)")
    together_layer = folium.FeatureGroup(name="Flagged cabinets shown in their cluster", show=False)
    info = clusters.set_index("cluster_id")

    for cid, shape in areas.items():
        p = info.loc[cid]
        folium.GeoJson({"type": "Feature", "geometry": mapping(shape), "properties": {}},
                       style_function=lambda _, c=colours[cid]: {"fillColor": c, "fillOpacity": 0.15,
                                                                 "color": "#555555", "weight": 0.8},
                       tooltip=f"{cid}: team of {p['people']}, {demand_text(p['demand_per_day'])} "
                               f"({p['fill_rate']:.0%} full), "
                               f"{p['n_cabinets']} cabinets").add_to(territory_layer)

    for cid, p in info.iterrows():
        special = p["demand_rule"] == "one person, special case"
        members = np.where((cluster_of == cid) & ~flagged)[0]
        if len(members):
            band = shapely.transform(outline(members).buffer(0.2), unproject_km)
            dash = None if p["people"] == people else "8 6"    # dashed outline for a smaller team
            folium.GeoJson({"type": "Feature", "geometry": mapping(band), "properties": {}},
                           style_function=lambda _, c=colours[cid], d=dash: {"fillColor": c, "fillOpacity": 0.1,
                                                                             "color": c, "weight": 2.5,
                                                                             "dashArray": d},
                           tooltip=f"outline of {cid} (team of {p['people']})").add_to(outline_layer)
        every = np.where(cluster_of == cid)[0]
        if flagged[every].any():   # a dashed outline around all its cabinets, flagged ones included
            band = shapely.transform(outline(every).buffer(0.2), unproject_km)
            folium.GeoJson({"type": "Feature", "geometry": mapping(band), "properties": {}},
                           style_function=lambda _, c=colours[cid]: {"fillColor": c, "fillOpacity": 0.05,
                                                                     "color": c, "weight": 2,
                                                                     "dashArray": "4 6"},
                           tooltip=f"{cid} with its flagged cabinets (centre unchanged)").add_to(together_layer)
        folium.Circle([p["centre_lat"], p["centre_long"]], radius=radius_km * 1000, color=colours[cid],
                      weight=1, dash_array="6 6", fill=False).add_to(circle_layer)
        popup = (f"<b>Centre of {cid}</b><br><b>{people_text(p['people'])}</b><br>"
                 f"centre lat, long: {p['centre_lat']:.5f}, {p['centre_long']:.5f}<br>"
                 f"{p['n_cabinets']} cabinets<br>"
                 f"demand: {demand_text(p['demand_per_day'])} ({p['fill_rate']:.0%} full), "
                 f"{demand_text(p['demand_per_person'], unit='')} per person<br>"
                 f"farthest cabinet by road: {p['max_road_dist_to_centre_km']:.2f} km<br>"
                 f"snapped to road: {snap_km(p['centre_lat'], p['centre_long']) * 1000:.0f} m<br>"
                 f"team of {p['people']}: {p['min_demand']:g} to {p['max_demand']:g} tickets/day")
        if special:
            popup += f"<br><b>special case</b>: one person, up to {SOLO_MAX_DEMAND:g} tickets/day"
        elif p["demand_rule"] == "below minimum":
            popup += "<br><b>below minimum</b>: leftovers that no cluster had room for"
        if p["n_straight_line_fallback"] or p["n_leftover"]:
            popup += (f"<br>drawn apart, not used for the centre: {p['n_straight_line_fallback']} fallback, "
                      f"{p['n_leftover']} leftover")
        popup += (f"<br>secondary cabinets: {p['n_secondary']} "
                  f"({demand_text(p['secondary_demand_per_day'])})")
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=centre_icon(p["people"], colours[cid], special),
                      tooltip=f"{cid} : {people_text(p['people'])}", z_index_offset=1000,
                      popup=folium.Popup(popup, max_width=280)).add_to(centre_layer)
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=name_icon(cid, p["people"]),
                      interactive=False).add_to(name_layer)

    position = {cab: i for i, cab in enumerate(cabinets["CabinetId"])}
    secondary_for = secondaries.groupby("CabinetId")["cluster_id"].apply(lambda s: ", ".join(sorted(s)))
    notes = {"road": "",
             "straight-line fallback": "<br><b>straight-line fallback</b>: too far by road, placed with the nearest "
                                       f"cluster within {RADIUS_KM:g} km in a straight line",
             "leftover": "<br><b>leftover</b>: its own cluster was too small for a team, so it joined the nearest "
                         "cluster border"}
    for a in assignments.itertuples():
        i = position[a.CabinetId]
        p = info.loc[a.cluster_id]
        left = a.distance_rule != "road"
        popup = (f"<b>{a.CabinetId}</b><br>cluster: {a.cluster_id} ({people_text(a.people)})<br>"
                 f"demand: {demand_text(a.allocated_demand, 3)}<br>"
                 f"cabinet lat, long: {LAT[i]:.5f}, {LON[i]:.5f}<br>"
                 f"centre of {a.cluster_id}: {p['centre_lat']:.5f}, {p['centre_long']:.5f}<br>"
                 f"to cluster centre: {a.road_dist_to_centre_km:.2f} km by road, "
                 f"{a.straight_dist_to_centre_km:.2f} km straight line" + notes[a.distance_rule])
        if left:
            popup += (f". Its demand counts for {a.cluster_id}, but it is drawn apart and does not move the "
                      f"centre.")
        if a.CabinetId in secondary_for.index:
            popup += f"<br>secondary for: {secondary_for[a.CabinetId]}"
        colour, weight = RINGS[a.distance_rule]
        folium.CircleMarker([LAT[i], LON[i]], radius=3 + 2.5 * np.sqrt(DEM[i]), color=colour, weight=weight,
                            fill=True, fill_color=FLAGGED_FILL if left else colours[a.cluster_id], fill_opacity=0.9,
                            tooltip=f"{a.CabinetId}" + (f" ({a.distance_rule})" if left else ""),
                            popup=folium.Popup(popup, max_width=280)).add_to(flag_layer if left else dot_layer)
        if left:   # the same cabinet in its cluster's colour, for the "shown in their cluster" layer
            folium.CircleMarker([LAT[i], LON[i]], radius=3 + 2.5 * np.sqrt(DEM[i]), color=colour, weight=weight,
                                fill=True, fill_color=colours[a.cluster_id], fill_opacity=0.9,
                                tooltip=f"{a.CabinetId} ({a.distance_rule})",
                                popup=folium.Popup(popup, max_width=280)).add_to(together_layer)

    link_layer = folium.FeatureGroup(name="Secondary links", show=False)
    for s in secondaries.itertuples():
        p, i = info.loc[s.cluster_id], position[s.CabinetId]
        folium.PolyLine([[p["centre_lat"], p["centre_long"]], [LAT[i], LON[i]]], color=colours[s.cluster_id],
                        weight=1, opacity=0.7, dash_array="4 4",
                        tooltip=f"{s.CabinetId}: secondary for {s.cluster_id}").add_to(link_layer)

    for layer in (territory_layer, outline_layer, circle_layer, link_layer, dot_layer, flag_layer, together_layer,
                  name_layer, centre_layer):
        layer.add_to(fmap) #Made by Iman Thaqif - Intern @ NAS, August 26 - Oct 26.
    folium.LayerControl(collapsed=False).add_to(fmap)
    title = (f"<div style='position:fixed;bottom:24px;left:10px;z-index:9999;background:white;padding:6px 10px;"
             f"border-radius:4px;font:14px sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)'>"
             f"<b>{len(clusters)} clusters, {int(clusters['people'].sum())} people per day</b><br>"
             f"teams of {min_people} to {people} (number on the centre = team size), "
             f"{radius_km:g} km by road from the centre ({2 * radius_km:g} km across), road first</div>")
    fmap.get_root().html.add_child(folium.Element(title))
    return fmap


fmap = make_map(clusters, assignments, secondaries, RADIUS_KM, PEOPLE_PER_CLUSTER, MIN_PEOPLE)
fmap.save(OUT_DIR / "map.html")
print(f"saved {OUTPUT_FOLDER}/map.html")
fmap

## Notes and limitations
* **Road distances** come from OSRM on OpenStreetMap: the length of the fastest route, without traffic. Missing or
  wrong roads give wrong distances. They are measured from where a point meets the nearest road.
* **OSRM server.** `OSRM_URL` points at our own server on this laptop (Malaysia extract from Geofabrik); start it
  with `start_osrm.bat`. No coordinates leave the laptop. Download a new extract from time to time. The public demo
  server `router.project-osrm.org` also works (with `OSRM_PAUSE_S = 1.0`), but it sends coordinates outside: mock
  data only.
* **Cache.** Saved distances are keyed by coordinates. After changing the server or the map data, delete the cache
  file so the distances are fetched again.
* **Outlines** use cabinet positions only: two clusters with clear outlines can still be split by a river or an
  expressway.
* **Rules.** Teams, tickets per person and the radius are in `parameters.csv`. Set `MIN_PEOPLE = 2` if nobody
  should work alone, and leave `SOLO_MAX_DEMAND` blank to switch off the one-person special case. A smaller radius
  means more straight-line fallbacks, since roads are often about twice the straight line.
* **Smaller teams use what is left**: clusters are built for full teams first, and quiet areas are not rebuilt from
  scratch for small teams.
* `demand` is an average. Busy days will exceed `CAP`; the fill rate is a planning figure.
* **Cluster IDs** are numbered on every run, so the same area can get another ID after a data change.
* **Secondary cabinets** are a list for the manager, not part of the plan.
* **Scaled demand.** With a large scale-up, the lightest clusters have only `MIN_DEMAND_PER_PERSON / DEMAND_SCALE`
  real tickets per person: check *Real tickets per person* in the summary.